In [ ]:
# Re-download AHD, Arabic-SQuAD, ARCD
import json
import urllib.request
from pathlib import Path

RAW = Path('/content/pstgen/raw')
RAW.mkdir(parents=True, exist_ok=True)

def already(path, min_bytes=10_000):
    p = Path(path)
    return p.exists() and p.stat().st_size > min_bytes


# 1) Health: medinstruct-52k-arabic
ahd_target = RAW / 'AHD.csv'
if already(ahd_target, min_bytes=1_000_000):
    print(f'✓ AHD.csv already present ({ahd_target.stat().st_size/1e6:.1f} MB)')
else:
    print('[1/3] downloading medinstruct-52k-arabic...')
    from datasets import load_dataset
    import pandas as pd
    ds = load_dataset('adlbh/medinstruct-52k-arabic', split='train')
    df = ds.to_pandas()
    def _q(row):
        q = str(row.get('instruction') or '').strip()
        extra = str(row.get('input') or '').strip()
        return (q + ' ' + extra).strip() if extra else q
    out = pd.DataFrame({
        'Question': df.apply(_q, axis=1),
        'Answer':   df['output'].astype(str),
        'Category': 'medical',
    })
    out = out[out['Question'].str.len() > 5].reset_index(drop=True)
    out.to_csv(ahd_target, index=False)
    print(f'✓ saved AHD.csv ({ahd_target.stat().st_size/1e6:.1f} MB, {len(out):,} rows)')

# 2) Arabic-SQuAD
soqal_target = RAW / 'Arabic-SQuAD.json'
if already(soqal_target):
    print(f'✓ Arabic-SQuAD.json already present')
else:
    print('[2/3] downloading Arabic-SQuAD...')
    url = 'https://raw.githubusercontent.com/husseinmozannar/SOQAL/master/data/Arabic-SQuAD.json'
    urllib.request.urlretrieve(url, soqal_target)
    print(f'✓ saved Arabic-SQuAD.json ({soqal_target.stat().st_size/1e6:.1f} MB)')

# 3) ARCD
arcd_target = RAW / 'ARCD.json'
if already(arcd_target):
    print(f'✓ ARCD.json already present')
else:
    print('[3/3] downloading ARCD...')
    url = 'https://raw.githubusercontent.com/husseinmozannar/SOQAL/master/data/arcd.json'
    urllib.request.urlretrieve(url, arcd_target)
    print(f'✓ saved ARCD.json ({arcd_target.stat().st_size/1e6:.1f} MB)')

print('\n✓ all datasets ready in /content/pstgen/raw/')

[1/3] downloading medinstruct-52k-arabic...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/46.9M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/52002 [00:00<?, ? examples/s]

✓ saved AHD.csv (100.2 MB, 52,002 rows)
[2/3] downloading Arabic-SQuAD...
✓ saved Arabic-SQuAD.json (51.3 MB)
[3/3] downloading ARCD...
✓ saved ARCD.json (1.9 MB)

✓ all datasets ready in /content/pstgen/raw/


In [ ]:
# Compute exact length statistics for Section 3.2 dataset description
import numpy as np
import pandas as pd
import json
from pathlib import Path

RAW = Path('/content/pstgen/raw')

# ============================================================
# Health corpus: medinstruct-52k-arabic (AHD.csv)
# Columns: Question, Answer, Category
# ============================================================
print("=" * 60)
print("HEALTH CORPUS — medinstruct-52k-arabic (AHD.csv)")
print("=" * 60)

df_h = pd.read_csv(RAW / 'AHD.csv')
print(f"Total rows: {len(df_h):,}")
print(f"Columns: {df_h.columns.tolist()}")

texts = df_h['Question'].fillna('').astype(str)
char_lens = texts.str.len()
word_lens = texts.str.split().str.len()

print(f"\nQuestion length (chars): mean={char_lens.mean():.1f}, median={char_lens.median():.0f}, "
      f"p25/p75={char_lens.quantile(0.25):.0f}/{char_lens.quantile(0.75):.0f}, min/max={char_lens.min()}/{char_lens.max()}")
print(f"Question length (words): mean={word_lens.mean():.1f}, median={word_lens.median():.0f}, "
      f"p25/p75={word_lens.quantile(0.25):.0f}/{word_lens.quantile(0.75):.0f}, min/max={word_lens.min()}/{word_lens.max()}")

# Vocabulary on a 10K sample
sample = texts.sample(min(10000, len(texts)), random_state=42)
tok_h = []
for t in sample:
    tok_h.extend(t.split())
vocab_h = len(set(tok_h))
print(f"Approx vocabulary (10K sample, whitespace tokens): {vocab_h:,}")

health_stats = {
    "n_texts": int(len(df_h)),
    "char_mean": float(char_lens.mean()),
    "char_median": float(char_lens.median()),
    "char_min": int(char_lens.min()),
    "char_max": int(char_lens.max()),
    "char_p25": float(char_lens.quantile(0.25)),
    "char_p75": float(char_lens.quantile(0.75)),
    "word_mean": float(word_lens.mean()),
    "word_median": float(word_lens.median()),
    "word_min": int(word_lens.min()),
    "word_max": int(word_lens.max()),
    "word_p25": float(word_lens.quantile(0.25)),
    "word_p75": float(word_lens.quantile(0.75)),
    "vocab_sample_10k": vocab_h,
}


# ============================================================
# Education corpus: SOQAL Arabic-SQuAD + ARCD
# ============================================================
print("\n" + "=" * 60)
print("EDUCATION CORPUS — SOQAL Arabic-SQuAD + ARCD")
print("=" * 60)

with open(RAW / 'Arabic-SQuAD.json', 'r', encoding='utf-8') as f:
    sq = json.load(f)

sq_paragraphs, sq_questions = [], []
for article in sq['data']:
    for para in article['paragraphs']:
        sq_paragraphs.append(para['context'])
        for qa in para['qas']:
            sq_questions.append(qa['question'])
print(f"Arabic-SQuAD: {len(sq['data'])} articles, {len(sq_paragraphs):,} paragraphs, {len(sq_questions):,} questions")

with open(RAW / 'ARCD.json', 'r', encoding='utf-8') as f:
    arcd = json.load(f)
arcd_paragraphs, arcd_questions = [], []
for article in arcd['data']:
    for para in article['paragraphs']:
        arcd_paragraphs.append(para['context'])
        for qa in para['qas']:
            arcd_questions.append(qa['question'])
print(f"ARCD: {len(arcd['data'])} articles, {len(arcd_paragraphs):,} paragraphs, {len(arcd_questions):,} questions")

all_paragraphs = sq_paragraphs + arcd_paragraphs
all_questions = sq_questions + arcd_questions

para_chars = pd.Series([len(p) for p in all_paragraphs])
para_words = pd.Series([len(p.split()) for p in all_paragraphs])
q_chars = pd.Series([len(q) for q in all_questions])
q_words = pd.Series([len(q.split()) for q in all_questions])

print(f"\nParagraph stats (n={len(all_paragraphs):,}):")
print(f"  chars: mean={para_chars.mean():.1f}, median={para_chars.median():.0f}, "
      f"p25/p75={para_chars.quantile(0.25):.0f}/{para_chars.quantile(0.75):.0f}, min/max={para_chars.min()}/{para_chars.max()}")
print(f"  words: mean={para_words.mean():.1f}, median={para_words.median():.0f}, "
      f"p25/p75={para_words.quantile(0.25):.0f}/{para_words.quantile(0.75):.0f}, min/max={para_words.min()}/{para_words.max()}")

print(f"\nQuestion stats (n={len(all_questions):,}):")
print(f"  chars: mean={q_chars.mean():.1f}, median={q_chars.median():.0f}")
print(f"  words: mean={q_words.mean():.1f}, median={q_words.median():.0f}")

# Vocabulary on 5K paragraph sample
import random
random.seed(42)
edu_sample = random.sample(all_paragraphs, min(5000, len(all_paragraphs)))
tok_e = []
for p in edu_sample:
    tok_e.extend(p.split())
vocab_e = len(set(tok_e))
print(f"\nApprox vocabulary (5K paragraph sample): {vocab_e:,}")

edu_stats = {
    "n_articles_squad": len(sq['data']),
    "n_articles_arcd": len(arcd['data']),
    "n_paragraphs": int(len(all_paragraphs)),
    "n_questions": int(len(all_questions)),
    "para_char_mean": float(para_chars.mean()),
    "para_char_median": float(para_chars.median()),
    "para_char_p25": float(para_chars.quantile(0.25)),
    "para_char_p75": float(para_chars.quantile(0.75)),
    "para_char_min": int(para_chars.min()),
    "para_char_max": int(para_chars.max()),
    "para_word_mean": float(para_words.mean()),
    "para_word_median": float(para_words.median()),
    "para_word_p25": float(para_words.quantile(0.25)),
    "para_word_p75": float(para_words.quantile(0.75)),
    "para_word_min": int(para_words.min()),
    "para_word_max": int(para_words.max()),
    "q_char_mean": float(q_chars.mean()),
    "q_word_mean": float(q_words.mean()),
    "vocab_sample_5k": vocab_e,
}


# ============================================================
# Save and summarize
# ============================================================
results_dir = Path('/content/pstgen/results')
results_dir.mkdir(parents=True, exist_ok=True)

with open(results_dir / 'dataset_stats.json', 'w') as f:
    json.dump({"health": health_stats, "education": edu_stats}, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 60)
print("KEY NUMBERS FOR SECTION 3.2 COMPARISON TABLE")
print("=" * 60)
print(f"\nHealth (medinstruct-52k-arabic):")
print(f"  n = {health_stats['n_texts']:,} texts")
print(f"  mean length = {health_stats['word_mean']:.1f} words ({health_stats['char_mean']:.0f} chars)")
print(f"  median length = {health_stats['word_median']:.0f} words ({health_stats['char_median']:.0f} chars)")

print(f"\nEducation (SOQAL+ARCD):")
print(f"  n = {edu_stats['n_paragraphs']:,} paragraphs across "
      f"{edu_stats['n_articles_squad'] + edu_stats['n_articles_arcd']} articles")
print(f"  + {edu_stats['n_questions']:,} questions")
print(f"  paragraph mean length = {edu_stats['para_word_mean']:.1f} words ({edu_stats['para_char_mean']:.0f} chars)")
print(f"  paragraph median length = {edu_stats['para_word_median']:.0f} words ({edu_stats['para_char_median']:.0f} chars)")

print(f"\n✓ saved to {results_dir / 'dataset_stats.json'}")

HEALTH CORPUS — medinstruct-52k-arabic (AHD.csv)
Total rows: 52,002
Columns: ['Question', 'Answer', 'Category']

Question length (chars): mean=215.5, median=196, p25/p75=101/288, min/max=24/3666
Question length (words): mean=35.8, median=33, p25/p75=16/48, min/max=4/572
Approx vocabulary (10K sample, whitespace tokens): 35,743

EDUCATION CORPUS — SOQAL Arabic-SQuAD + ARCD
Arabic-SQuAD: 231 articles, 10,364 paragraphs, 48,344 questions
ARCD: 155 articles, 465 paragraphs, 1,395 questions

Paragraph stats (n=10,829):
  chars: mean=594.3, median=552, p25/p75=407/734, min/max=91/2596
  words: mean=110.7, median=102, p25/p75=75/137, min/max=16/524

Question stats (n=49,739):
  chars: mean=50.4, median=47
  words: mean=9.1, median=9

Approx vocabulary (5K paragraph sample): 66,081

KEY NUMBERS FOR SECTION 3.2 COMPARISON TABLE

Health (medinstruct-52k-arabic):
  n = 52,002 texts
  mean length = 35.8 words (216 chars)
  median length = 33 words (196 chars)

Education (SOQAL+ARCD):
  n = 10,829 

# PST-Gen Final 2026 — Complete Colab Pipeline

**Path 1 paper**: *PST-Gen: A Principled Synthetic Timeline Generation Framework for Privacy-Preserving Cross-Domain Cascade Detection in Arabic NLP*

**Target venue:** IEEE Access (Q1)

## How to run

1. Runtime → Change runtime type → **GPU (T4)**
2. Run cells top to bottom. If Colab disconnects, re-run from Cell 1 — everything auto-resumes from Drive checkpoints.
3. Total time across sessions: ~6–8 hours.

## What's new vs. the previous notebook

- **AraModernBERT** replaces AraBERT (2026-current encoder, Dec 2024)
- **Register-adapted labeler** fixes the bad 87%/0.2% label distribution from the previous run
- **Custom Mamba-style event model** added as modern architectural baseline (2024 SSM)
- **AraModernBERT MonoBERT** as non-temporal sanity check
- **Dropped**: the Ensemble model (not defensible as a 2026 contribution)

## Final model set (5 models)

| # | Model | Role | Expected time |
|---|---|---|---|
| 1 | **AraModernBERT + BiLSTM** | Hero | ~1 h |
| 2 | **AraModernBERT + TCDT** | Transformer baseline | ~1 h |
| 3 | **Mamba-style Event SSM** | 2024 modern architecture | ~1 h |
| 4 | **AraModernBERT MonoBERT** | Non-temporal ablation | ~45 min |
| 5 | Jais-1.3B zero-shot | LLM baseline (optional — separate notebook) | - |


---
## 1. Setup — Drive, GPU, packages

In [ ]:
# Mount Drive and set up project layout
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/pstgen')
LOCAL_ROOT = Path('/content/pstgen')

for sub in ['raw', 'processed', 'synthetic', 'checkpoints', 'results', 'hf_cache']:
    (DRIVE_ROOT / sub).mkdir(parents=True, exist_ok=True)

LOCAL_ROOT.mkdir(exist_ok=True)
for sub in ['raw', 'processed', 'synthetic', 'checkpoints', 'results']:
    local_sub = LOCAL_ROOT / sub
    drive_sub = DRIVE_ROOT / sub
    if local_sub.exists() and not local_sub.is_symlink():
        try:
            local_sub.rmdir()
        except OSError:
            pass
    if not local_sub.exists():
        local_sub.symlink_to(drive_sub)

os.environ['HF_HOME'] = str(DRIVE_ROOT / 'hf_cache')
os.environ['TRANSFORMERS_CACHE'] = str(DRIVE_ROOT / 'hf_cache')

print('✓ project root:', LOCAL_ROOT)
print('✓ drive root:  ', DRIVE_ROOT)

Mounted at /content/drive
✓ project root: /content/pstgen
✓ drive root:   /content/drive/MyDrive/pstgen


In [ ]:
# GPU check
import torch
print(f'torch   {torch.__version__}')
print(f'cuda    {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'device  {torch.cuda.get_device_name(0)}')
    print(f'memory  {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠ no GPU detected. Runtime → Change runtime type → GPU (T4).')

torch   2.10.0+cu128
cuda    True
device  Tesla T4
memory  15.6 GB


In [ ]:
# Install packages
# transformers 4.48+ is needed for AraModernBERT (ModernBertModel)
!pip install -q 'transformers>=4.48.0' 'datasets>=2.19.0' tokenizers tqdm 'pyyaml>=6.0' \
     arabic-reshaper python-bidi sentencepiece
print('✓ packages installed')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.6/300.6 kB 21.8 MB/s eta 0:00:00
✓ packages installed


---
## 2. Download source datasets (no auth)

In [ ]:
# Download AHD (medinstruct-52k-arabic), SOQAL Arabic-SQuAD, ARCD
# All public, no auth required. Idempotent (safe to re-run).

import json
import urllib.request
from pathlib import Path

RAW = Path('/content/pstgen/raw')

def already(path, min_bytes=10_000):
    p = Path(path)
    return p.exists() and p.stat().st_size > min_bytes


# 1) Health corpus — medinstruct-52k-arabic (replaces AHD)
ahd_target = RAW / 'AHD.csv'
if already(ahd_target, min_bytes=1_000_000):
    print(f'✓ health dataset already present: {ahd_target.name} ({ahd_target.stat().st_size/1e6:.1f} MB)')
else:
    print('[1/3] downloading medinstruct-52k-arabic from HuggingFace...')
    from datasets import load_dataset
    import pandas as pd
    ds = load_dataset('adlbh/medinstruct-52k-arabic', split='train')
    df = ds.to_pandas()
    def _q(row):
        q = str(row.get('instruction') or '').strip()
        extra = str(row.get('input') or '').strip()
        return (q + ' ' + extra).strip() if extra else q
    out = pd.DataFrame({
        'Question': df.apply(_q, axis=1),
        'Answer':   df['output'].astype(str),
        'Category': 'medical',
    })
    out = out[out['Question'].str.len() > 5].reset_index(drop=True)
    out.to_csv(ahd_target, index=False)
    print(f'✓ saved {ahd_target.name} ({ahd_target.stat().st_size/1e6:.1f} MB, {len(out):,} rows)')


# 2) SOQAL Arabic-SQuAD
soqal_target = RAW / 'Arabic-SQuAD.json'
if already(soqal_target):
    print(f'✓ SOQAL already present: {soqal_target.name} ({soqal_target.stat().st_size/1e6:.1f} MB)')
else:
    urls = [
        'https://raw.githubusercontent.com/husseinmozannar/SOQAL/master/data/Arabic-SQuAD.json',
        'https://raw.githubusercontent.com/WissamAntoun/Arabic_QA_Datasets/master/data/Arabic-SQuAD.json',
    ]
    for url in urls:
        print(f'[2/3] trying {url} ...')
        try:
            urllib.request.urlretrieve(url, soqal_target)
            with open(soqal_target, 'r', encoding='utf-8') as f:
                data = json.load(f)
            n_qas = sum(len(p.get('qas', [])) for a in data.get('data', []) for p in a.get('paragraphs', []))
            print(f'✓ saved ({soqal_target.stat().st_size/1e6:.1f} MB, {n_qas:,} Q&A)')
            break
        except Exception as e:
            print(f'  failed: {e}')
            if soqal_target.exists():
                soqal_target.unlink()


# 3) ARCD (supplementary)
arcd_target = RAW / 'ARCD.json'
if already(arcd_target):
    print(f'✓ ARCD already present: {arcd_target.name}')
else:
    url = 'https://raw.githubusercontent.com/husseinmozannar/SOQAL/master/data/arcd.json'
    try:
        urllib.request.urlretrieve(url, arcd_target)
        print(f'✓ saved {arcd_target.name} ({arcd_target.stat().st_size/1e6:.1f} MB)')
    except Exception as e:
        print(f'  failed: {e}')

print('\n=== SUMMARY ===')
for p in [ahd_target, soqal_target, arcd_target]:
    tag = '✓' if p.exists() else '✗'
    size = f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else 'MISSING'
    print(f'  {tag} {p.name:25s}  {size}')


✓ health dataset already present: AHD.csv (100.2 MB)
✓ SOQAL already present: Arabic-SQuAD.json (51.3 MB)
✓ ARCD already present: ARCD.json

=== SUMMARY ===
  ✓ AHD.csv                    100.2 MB
  ✓ Arabic-SQuAD.json          51.3 MB
  ✓ ARCD.json                  1.9 MB


---
## 3. Library code (inlined — self-contained)

In [ ]:
%%writefile /content/pstgen_lib.py
"""PST-Gen library (final 2026 version) — to be inlined into the notebook.
All classes defined here get written to /content/pstgen_lib.py inside Colab.
"""
from __future__ import annotations

import json
import re
import time
import unicodedata
from collections import defaultdict
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Iterable, Optional, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer


# ============================================================================
# Arabic preprocessing
# ============================================================================
class ArabicTextPreprocessor:
    """Normalize Arabic text: strip diacritics, normalize alif/yaa/taa marbuta."""
    ARABIC_DIACRITICS_RE = re.compile(r"[\u064B-\u065F\u0670]")
    TATWEEL_RE = re.compile(r"\u0640")
    REPEATED_CHAR_RE = re.compile(r"(.)\1{2,}")
    WHITESPACE_RE = re.compile(r"\s+")
    NORMALIZATION_MAP = {
        "\u0622": "\u0627",  # آ -> ا
        "\u0623": "\u0627",  # أ -> ا
        "\u0625": "\u0627",  # إ -> ا
        "\u0649": "\u064A",  # ى -> ي
        "\u0629": "\u0647",  # ة -> ه
    }

    def __init__(self, min_length: int = 20, max_length: int = 512):
        self.min_length = min_length
        self.max_length = max_length

    def normalize(self, text: str) -> str:
        if not isinstance(text, str) or not text.strip():
            return ""
        text = unicodedata.normalize("NFKC", text)
        text = self.ARABIC_DIACRITICS_RE.sub("", text)
        text = self.TATWEEL_RE.sub("", text)
        for src, tgt in self.NORMALIZATION_MAP.items():
            text = text.replace(src, tgt)
        text = self.REPEATED_CHAR_RE.sub(r"\1", text)
        text = self.WHITESPACE_RE.sub(" ", text).strip()
        return text

    def is_valid(self, text: str) -> bool:
        return bool(text) and self.min_length <= len(text) <= self.max_length

    def filter_and_normalize(self, texts: Iterable[str]) -> list[str]:
        out = []
        for t in texts:
            n = self.normalize(t)
            if self.is_valid(n):
                out.append(n)
        return out


# ============================================================================
# Register-adapted heuristic labeler (for medinstruct + SOQAL)
# ============================================================================
class RegisterAdaptedHeuristics:
    """Risk labeler tuned for formal/clinical Arabic (medinstruct) and
    factual/encyclopedic Arabic (SOQAL).
    """

    HEALTH_SEVERITY_PATTERNS = [
        (r"اعاني|أعاني", 4),
        (r"اشعر ب|أشعر ب", 3),
        (r"عندي الم|عندي ألم|يؤلمني|يؤلمنى", 3),
        (r"لا استطيع|لا أستطيع|لم استطع|لم أستطع", 2),
        (r"شديد|حاد|مزمن|مستمر|متواصل|متكرر", 3),
        (r"خطير|خطر|حرج|طارئ|عاجل", 3),
        (r"لا يحتمل|لا أتحمل|لا اتحمل", 3),
        (r"اكتئاب|مكتئب|قلق شديد|توتر شديد|ارهاق|إرهاق", 4),
        (r"انتحار|ايذاء الذات|يأس", 5),
        (r"ارق|أرق|نوم متقطع|صعوبه في النوم|صعوبة في النوم", 3),
        (r"نزيف|فقدان الوعي|اغماء|إغماء|تشنج", 4),
        (r"ضيق تنفس|صعوبة تنفس|صعوبه تنفس|خفقان", 3),
        (r"دوار|دوخه|دوخة|غثيان شديد", 2),
        (r"منذ\s+(?:اسابيع|أسابيع|شهور|شهر|سنوات|سنه|سنة|فتره طويله|فترة طويلة)", 3),
        (r"تم تشخيص|شخصت|مصاب ب|اعاني من مرض", 3),
    ]

    HEALTH_INFORMATIONAL_PATTERNS = [
        (r"^اشرح|^اكتب|^قدم|^عرف|^لخص|^اذكر|^ناقش|^قارن", 3),
        (r"ما هو|ما هي|ما الفرق|كيف يعمل|كيف يتم", 2),
        (r"قدم ملخص|قدم نبذه|قدم نبذة|قدم معلومات", 3),
        (r"<noinput>|noinput|<input>", 2),
        (r"الموصوف طبيا|الموصوف طبي ا", 2),
    ]

    EDU_STRUGGLE_PATTERNS = [
        (r"تسرب|تسرب دراسي|انقطاع|انقطع|غياب|تغيب|غبت", 4),
        (r"ترك الدراسه|ترك الدراسة|لم يكمل|لم تكمل", 4),
        (r"فشل|فشلت|رسب|رسبت|راسب|اخفاق|إخفاق", 4),
        (r"صعوبه في|صعوبة في|صعب على|صعب علي", 3),
        (r"لم يستطع|لم يتمكن|لم استطع|لم اتمكن", 3),
        (r"ضعف في|ضعيف في|متأخر في|متاخر في", 3),
        (r"قلق الامتحان|قلق الاختبار|خوف من الامتحان|توتر الامتحان", 4),
        (r"ضغط دراسي|ضغط مدرسي|ارهاق دراسي", 4),
        (r"صعوبات تعلم|عسر قراءه|عسر قراءة|عسر كتابه", 4),
        (r"اضطراب فرط الحركه|فرط نشاط|تشتت الانتباه", 3),
        (r"لا افهم|لا أفهم|لم افهم|لم أفهم|لا اتذكر|لا أتذكر", 3),
        (r"مش فاهم|مش قادر افهم|ما فهمت", 3),
    ]

    EDU_FACTUAL_PATTERNS = [
        (r"^ما هو|^ما هي|^متى|^اين|^أين|^من هو|^من هي|^كم|^كيف|^لماذا|^هل", 3),
        (r"سنة \d{3,4}|عام \d{3,4}|تاريخ|قرن|ميلادي|هجري", 2),
        (r"يعتبر|يعد|تأسس|اسس|تم تأسيس|يقع في|تقع في", 2),
        (r"يبلغ عدد|عدد السكان|المساحه|المساحة", 2),
    ]

    def _score(self, text: str, patterns) -> float:
        score = 0.0
        for pat, weight in patterns:
            score += len(re.findall(pat, text)) * weight
        return score

    def label(self, text: str, domain: str) -> str:
        text = text.strip()
        if not text:
            return "NoRisk"
        if domain == "Health":
            sev = self._score(text, self.HEALTH_SEVERITY_PATTERNS)
            inf = self._score(text, self.HEALTH_INFORMATIONAL_PATTERNS)
            if sev >= 4 and sev > inf:
                return "HealthConcern"
            if sev >= 6:
                return "HealthConcern"
            return "NoRisk"
        else:
            stg = self._score(text, self.EDU_STRUGGLE_PATTERNS)
            fac = self._score(text, self.EDU_FACTUAL_PATTERNS)
            if stg >= 4 and stg > fac:
                return "EduDisengage"
            if stg >= 6:
                return "EduDisengage"
            return "NoRisk"


# ============================================================================
# PST-Gen semi-Markov timeline generator
# ============================================================================
@dataclass
class PSTGenConfig:
    num_timelines: int = 10_000
    events_per_timeline: int = 20
    health_prevalence_target: float = 0.30
    edu_prevalence_target: float = 0.05
    domain_transition_h_to_e: float = 0.35
    domain_transition_e_to_h: float = 0.31
    cascade_window: int = 5
    cascade_prob_in_window: float = 0.25
    p_health_initial: float = 0.65
    seed: int = 42


class PSTGenGenerator:
    def __init__(
        self,
        health_texts_by_label: dict[str, list[str]],
        edu_texts_by_label: dict[str, list[str]],
        cfg: PSTGenConfig,
    ):
        self.health = health_texts_by_label
        self.edu = edu_texts_by_label
        self.cfg = cfg
        self.rng = np.random.default_rng(cfg.seed)

    def _sample_text(self, pool: dict[str, list[str]], label: str) -> str:
        texts = pool.get(label, [])
        if not texts:
            for lab, ts in pool.items():
                if ts:
                    texts = ts
                    break
        if not texts:
            return "[empty]"
        return texts[self.rng.integers(0, len(texts))]

    def _sample_domain(self, prev: Optional[str]) -> str:
        if prev is None:
            return "Health" if self.rng.random() < self.cfg.p_health_initial else "Education"
        if prev == "Health":
            return "Education" if self.rng.random() < self.cfg.domain_transition_h_to_e else "Health"
        return "Health" if self.rng.random() < self.cfg.domain_transition_e_to_h else "Education"

    def _sample_risk(self, domain: str) -> str:
        if domain == "Health":
            return "HealthConcern" if self.rng.random() < self.cfg.health_prevalence_target else "NoRisk"
        return "EduDisengage" if self.rng.random() < self.cfg.edu_prevalence_target else "NoRisk"

    def _generate_one(self, citizen_id: str) -> dict:
        events = []
        risk_history: list[tuple[int, str, str]] = []
        prev_domain: Optional[str] = None
        for t in range(self.cfg.events_per_timeline):
            d = self._sample_domain(prev_domain)
            r = self._sample_risk(d)
            cascade = 0
            if r != "NoRisk":
                for (pos, pd_, pr) in risk_history:
                    if pd_ != d and (t - pos) <= self.cfg.cascade_window and pr != "NoRisk":
                        if self.rng.random() < self.cfg.cascade_prob_in_window:
                            cascade = 1
                            break
                risk_history.append((t, d, r))
            pool = self.health if d == "Health" else self.edu
            text = self._sample_text(pool, r)
            events.append({
                "position": t, "domain": d, "text": text,
                "risk_label": r, "cascade": cascade,
            })
            prev_domain = d
        return {"citizen_id": citizen_id, "events": events}

    def generate(self) -> list[dict]:
        out = []
        for i in tqdm(range(self.cfg.num_timelines), desc="generating"):
            out.append(self._generate_one(f"citizen_{i:06d}"))
        return out


# ============================================================================
# Dataset
# ============================================================================
RISK_LABELS = {"NoRisk": 0, "HealthConcern": 1, "EduDisengage": 2}
DOMAIN_LABELS = {"Health": 0, "Education": 1}


class TimelineDataset(Dataset):
    def __init__(
        self,
        path: str | Path,
        tokenizer_name: str = "NAMAA-Space/AraModernBert-Base-V1.0",
        max_events: int = 20,
        max_tokens_per_event: int = 128,
    ):
        self.path = Path(path)
        self.tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
        self.max_events = max_events
        self.max_tokens = max_tokens_per_event
        with open(self.path, "r", encoding="utf-8") as f:
            self.timelines = json.load(f)

    def __len__(self):
        return len(self.timelines)

    def __getitem__(self, idx: int):
        tl = self.timelines[idx]
        events = tl["events"][: self.max_events]
        texts = [e["text"] for e in events]
        enc = self.tokenizer(
            texts, padding="max_length", truncation=True,
            max_length=self.max_tokens, return_tensors="pt",
        )
        input_ids = enc["input_ids"]
        attention_mask = enc["attention_mask"]
        n = len(events)
        if n < self.max_events:
            pad = self.max_events - n
            input_ids = torch.cat([input_ids, torch.zeros(pad, self.max_tokens, dtype=torch.long)])
            attention_mask = torch.cat([attention_mask, torch.zeros(pad, self.max_tokens, dtype=torch.long)])
        domains = torch.full((self.max_events,), -1, dtype=torch.long)
        risk_labels = torch.full((self.max_events,), -1, dtype=torch.long)
        cascade_labels = torch.zeros(self.max_events, dtype=torch.float)
        event_mask = torch.zeros(self.max_events, dtype=torch.bool)
        for i, e in enumerate(events):
            domains[i] = DOMAIN_LABELS.get(e.get("domain", "Health"), 0)
            risk_labels[i] = RISK_LABELS.get(e.get("risk_label", "NoRisk"), 0)
            cascade_labels[i] = float(e.get("cascade", 0))
            event_mask[i] = True
        return {
            "input_ids": input_ids, "attention_mask": attention_mask,
            "domains": domains, "risk_labels": risk_labels,
            "cascade_labels": cascade_labels, "event_mask": event_mask,
            "citizen_id": tl.get("citizen_id", f"timeline_{idx:06d}"),
        }


def collate(batch):
    out = {}
    for k in ("input_ids","attention_mask","domains","risk_labels","cascade_labels","event_mask"):
        out[k] = torch.stack([b[k] for b in batch])
    out["citizen_id"] = [b["citizen_id"] for b in batch]
    return out


def build_loaders(ds, batch_size=16, train_frac=0.7, val_frac=0.15, seed=42):
    n = len(ds)
    n_train = int(n * train_frac); n_val = int(n * val_frac)
    n_test = n - n_train - n_val
    gen = torch.Generator().manual_seed(seed)
    tr, va, te = random_split(ds, [n_train, n_val, n_test], generator=gen)
    return (
        DataLoader(tr, batch_size=batch_size, shuffle=True, collate_fn=collate, num_workers=0),
        DataLoader(va, batch_size=batch_size, shuffle=False, collate_fn=collate, num_workers=0),
        DataLoader(te, batch_size=batch_size, shuffle=False, collate_fn=collate, num_workers=0),
    )


# ============================================================================
# Model configs (all use AraModernBERT by default — 2026 upgrade)
# ============================================================================
ARAMODERNBERT = "NAMAA-Space/AraModernBert-Base-V1.0"


@dataclass
class BiLSTMConfig:
    encoder_name: str = ARAMODERNBERT
    encoder_dim: int = 768
    hidden_dim: int = 128
    num_layers: int = 2
    dropout: float = 0.3
    num_risk_classes: int = 3
    freeze_encoder: bool = True
    max_events: int = 20


class BiLSTMCascadeDetector(nn.Module):
    """Hero model: frozen AraModernBERT -> BiLSTM -> dual heads."""

    def __init__(self, cfg: BiLSTMConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder = AutoModel.from_pretrained(cfg.encoder_name)
        if cfg.freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
            self.encoder.eval()
        self.lstm = nn.LSTM(
            input_size=cfg.encoder_dim, hidden_size=cfg.hidden_dim,
            num_layers=cfg.num_layers, batch_first=True, bidirectional=True,
            dropout=cfg.dropout if cfg.num_layers > 1 else 0.0,
        )
        self.layer_norm = nn.LayerNorm(2 * cfg.hidden_dim)
        self.dropout = nn.Dropout(cfg.dropout)
        self.risk_head = nn.Linear(2 * cfg.hidden_dim, cfg.num_risk_classes)
        self.cascade_head = nn.Linear(2 * cfg.hidden_dim, 1)

    def forward(self, input_ids, attention_mask, **unused):
        B, N, T = input_ids.shape
        flat_ids = input_ids.view(B * N, T)
        flat_mask = attention_mask.view(B * N, T)
        if self.cfg.freeze_encoder:
            with torch.no_grad():
                enc = self.encoder(input_ids=flat_ids, attention_mask=flat_mask)
        else:
            enc = self.encoder(input_ids=flat_ids, attention_mask=flat_mask)
        # AraModernBERT: use mean pooling of non-padding tokens (no [CLS] pooler)
        last_hidden = enc.last_hidden_state  # (B*N, T, D)
        mask = flat_mask.unsqueeze(-1).float()
        pooled = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        event_emb = pooled.view(B, N, self.cfg.encoder_dim)
        lstm_out, _ = self.lstm(event_emb)
        lstm_out = self.dropout(self.layer_norm(lstm_out))
        return {
            "risk_logits": self.risk_head(lstm_out),
            "cascade_logits": self.cascade_head(lstm_out).squeeze(-1),
            "event_emb": event_emb,
            "lstm_out": lstm_out,
        }

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable, "frozen": total - trainable}


@dataclass
class MonoBERTConfig:
    encoder_name: str = ARAMODERNBERT
    encoder_dim: int = 768
    num_risk_classes: int = 3
    dropout: float = 0.1


class MonoBERTCascadeDetector(nn.Module):
    """Non-temporal baseline: each event classified independently."""
    def __init__(self, cfg: MonoBERTConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder = AutoModel.from_pretrained(cfg.encoder_name)
        self.dropout = nn.Dropout(cfg.dropout)
        self.risk_head = nn.Linear(cfg.encoder_dim, cfg.num_risk_classes)
        self.cascade_head = nn.Linear(cfg.encoder_dim, 1)

    def forward(self, input_ids, attention_mask, **unused):
        B, N, T = input_ids.shape
        flat_ids = input_ids.view(B * N, T); flat_mask = attention_mask.view(B * N, T)
        enc = self.encoder(input_ids=flat_ids, attention_mask=flat_mask)
        mask = flat_mask.unsqueeze(-1).float()
        pooled = (enc.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        pooled = self.dropout(pooled)
        risk = self.risk_head(pooled).view(B, N, self.cfg.num_risk_classes)
        cascade = self.cascade_head(pooled).view(B, N)
        return {"risk_logits": risk, "cascade_logits": cascade}

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable, "frozen": total - trainable}


@dataclass
class TCDTConfig:
    encoder_name: str = ARAMODERNBERT
    encoder_dim: int = 768
    num_heads: int = 4
    ff_dim: int = 1536
    num_layers: int = 1
    dropout: float = 0.1
    max_events: int = 20
    num_risk_classes: int = 3
    freeze_encoder: bool = True


class TCDTModel(nn.Module):
    """Temporal Cross-Domain Transformer: frozen encoder + transformer over events."""
    def __init__(self, cfg: TCDTConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder = AutoModel.from_pretrained(cfg.encoder_name)
        if cfg.freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
            self.encoder.eval()
        self.pos_emb = nn.Embedding(cfg.max_events, cfg.encoder_dim)
        layer = nn.TransformerEncoderLayer(
            d_model=cfg.encoder_dim, nhead=cfg.num_heads,
            dim_feedforward=cfg.ff_dim, dropout=cfg.dropout,
            batch_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(layer, num_layers=cfg.num_layers)
        self.risk_head = nn.Linear(cfg.encoder_dim, cfg.num_risk_classes)
        self.cascade_head = nn.Linear(cfg.encoder_dim, 1)

    def forward(self, input_ids, attention_mask, **unused):
        B, N, T = input_ids.shape
        flat_ids = input_ids.view(B * N, T); flat_mask = attention_mask.view(B * N, T)
        if self.cfg.freeze_encoder:
            with torch.no_grad():
                enc = self.encoder(input_ids=flat_ids, attention_mask=flat_mask)
        else:
            enc = self.encoder(input_ids=flat_ids, attention_mask=flat_mask)
        mask = flat_mask.unsqueeze(-1).float()
        pooled = (enc.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        event_emb = pooled.view(B, N, self.cfg.encoder_dim)
        pos = torch.arange(N, device=input_ids.device)
        event_emb = event_emb + self.pos_emb(pos).unsqueeze(0)
        h = self.transformer(event_emb)
        return {
            "risk_logits": self.risk_head(h),
            "cascade_logits": self.cascade_head(h).squeeze(-1),
        }

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable, "frozen": total - trainable}


# ============================================================================
# Mamba-style event-sequence model (from scratch — no CUDA deps)
# ============================================================================
class MambaEventBlock(nn.Module):
    """Simplified Mamba-style selective SSM block for event sequences.

    Implements the core selective state-space mechanism from Gu & Dao (2023)
    adapted to short event sequences (N=20). Uses diagonal state transitions
    and parallel scan via cumulative products. Fully differentiable in PyTorch
    without requiring custom CUDA kernels — suitable for T4 free Colab.
    """
    def __init__(self, d_model: int, d_state: int = 16, expand: int = 2):
        super().__init__()
        self.d_model = d_model
        self.d_inner = d_model * expand
        self.d_state = d_state
        # Input projection (x -> [hidden, gate])
        self.in_proj = nn.Linear(d_model, 2 * self.d_inner, bias=False)
        # Selective parameters: B, C, Delta are input-dependent
        self.x_proj = nn.Linear(self.d_inner, d_state * 2 + self.d_inner, bias=False)
        # Initialize A as negative log-space for stability
        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).repeat(self.d_inner, 1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(self.d_inner))
        # Output projection
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        """x: (B, N, d_model)"""
        B, N, D = x.shape
        residual = x
        x = self.norm(x)
        # Dual projection -> hidden and gate
        xz = self.in_proj(x)  # (B, N, 2*d_inner)
        x_in, z = xz.chunk(2, dim=-1)
        # Get selective B, C, Delta
        x_dbl = self.x_proj(x_in)  # (B, N, d_state + d_state + d_inner)
        B_sel, C_sel, Delta = torch.split(x_dbl, [self.d_state, self.d_state, self.d_inner], dim=-1)
        Delta = F.softplus(Delta)  # (B, N, d_inner) — positive timesteps
        # Discretize: A_bar = exp(Delta * A), B_bar = Delta * B
        A = -torch.exp(self.A_log)  # (d_inner, d_state) — negative for stability
        # Recurrent scan: serial (for N=20 this is fine)
        h = torch.zeros(B, self.d_inner, self.d_state, device=x.device, dtype=x.dtype)
        outputs = []
        for t in range(N):
            # A_bar_t: (B, d_inner, d_state) = exp(Delta_t * A)
            A_bar = torch.exp(Delta[:, t, :].unsqueeze(-1) * A.unsqueeze(0))
            # B_bar_t: (B, d_inner, d_state)
            B_bar = Delta[:, t, :].unsqueeze(-1) * B_sel[:, t, :].unsqueeze(1)
            h = A_bar * h + B_bar * x_in[:, t, :].unsqueeze(-1)
            # y_t = C_t * h_t, summed over state
            y_t = torch.einsum('bis,bs->bi', h, C_sel[:, t, :])
            outputs.append(y_t + self.D * x_in[:, t, :])
        y = torch.stack(outputs, dim=1)  # (B, N, d_inner)
        # Gate
        y = y * F.silu(z)
        # Output projection
        y = self.out_proj(y)
        return y + residual


@dataclass
class MambaEventConfig:
    encoder_name: str = ARAMODERNBERT
    encoder_dim: int = 768
    d_state: int = 16
    expand: int = 2
    num_blocks: int = 2
    num_risk_classes: int = 3
    max_events: int = 20
    freeze_encoder: bool = True


class MambaEventModel(nn.Module):
    """Modern architectural baseline: AraModernBERT frozen -> Mamba-style SSM blocks.

    Represents the 2024-era state-space modeling paradigm for temporal sequences,
    providing a non-transformer, non-RNN architectural comparison.
    """
    def __init__(self, cfg: MambaEventConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder = AutoModel.from_pretrained(cfg.encoder_name)
        if cfg.freeze_encoder:
            for p in self.encoder.parameters():
                p.requires_grad = False
            self.encoder.eval()
        self.blocks = nn.ModuleList([
            MambaEventBlock(cfg.encoder_dim, d_state=cfg.d_state, expand=cfg.expand)
            for _ in range(cfg.num_blocks)
        ])
        self.risk_head = nn.Linear(cfg.encoder_dim, cfg.num_risk_classes)
        self.cascade_head = nn.Linear(cfg.encoder_dim, 1)

    def forward(self, input_ids, attention_mask, **unused):
        B, N, T = input_ids.shape
        flat_ids = input_ids.view(B * N, T); flat_mask = attention_mask.view(B * N, T)
        if self.cfg.freeze_encoder:
            with torch.no_grad():
                enc = self.encoder(input_ids=flat_ids, attention_mask=flat_mask)
        else:
            enc = self.encoder(input_ids=flat_ids, attention_mask=flat_mask)
        mask = flat_mask.unsqueeze(-1).float()
        pooled = (enc.last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-6)
        h = pooled.view(B, N, self.cfg.encoder_dim)
        for blk in self.blocks:
            h = blk(h)
        return {
            "risk_logits": self.risk_head(h),
            "cascade_logits": self.cascade_head(h).squeeze(-1),
        }

    def count_parameters(self):
        total = sum(p.numel() for p in self.parameters())
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {"total": total, "trainable": trainable, "frozen": total - trainable}


# ============================================================================
# Training
# ============================================================================
@dataclass
class TrainConfig:
    model_name: str = "bilstm"
    num_epochs: int = 20
    learning_rate: float = 3e-5
    weight_decay: float = 0.01
    grad_accum_steps: int = 1
    max_grad_norm: float = 1.0
    cascade_loss_weight: float = 1.5
    risk_class_weights: tuple = (0.49, 1.14, 12.02)
    patience: int = 3
    mixed_precision: bool = True
    checkpoint_dir: str = "/content/pstgen/checkpoints/bilstm"


class CascadeLoss(nn.Module):
    def __init__(self, risk_class_weights, cascade_weight):
        super().__init__()
        self.risk_weights = torch.tensor(risk_class_weights, dtype=torch.float)
        self.cascade_weight = cascade_weight
        self.risk_ce = nn.CrossEntropyLoss(weight=self.risk_weights, ignore_index=-1)
        self.cascade_bce = nn.BCEWithLogitsLoss(reduction="none")

    def forward(self, risk_logits, cascade_logits, risk_labels, cascade_labels, event_mask):
        if self.risk_ce.weight.device != risk_logits.device:
            self.risk_ce.weight = self.risk_ce.weight.to(risk_logits.device)
        B, N, C = risk_logits.shape
        risk_loss = self.risk_ce(risk_logits.reshape(-1, C), risk_labels.reshape(-1))
        cascade_elt = self.cascade_bce(cascade_logits, cascade_labels)
        m = event_mask.float()
        cascade_loss = (cascade_elt * m).sum() / m.sum().clamp(min=1.0)
        return {"total": risk_loss + self.cascade_weight * cascade_loss,
                "risk": risk_loss, "cascade": cascade_loss}


class Trainer:
    def __init__(self, model, train_loader, val_loader, cfg, device=None):
        self.model = model
        self.train_loader = train_loader; self.val_loader = val_loader; self.cfg = cfg
        self.device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.loss_fn = CascadeLoss(cfg.risk_class_weights, cfg.cascade_loss_weight)
        trainable = [p for p in model.parameters() if p.requires_grad]
        self.optim = AdamW(trainable, lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
        total_steps = max(1, len(train_loader) * cfg.num_epochs // max(cfg.grad_accum_steps, 1))
        self.sched = CosineAnnealingLR(self.optim, T_max=total_steps)
        self.scaler = GradScaler(enabled=cfg.mixed_precision and self.device.type == "cuda")
        self.ckpt_dir = Path(cfg.checkpoint_dir); self.ckpt_dir.mkdir(parents=True, exist_ok=True)
        self.start_epoch = 0; self.best_val_f1 = -1.0; self.patience_counter = 0; self.history = []

    def _ckpt(self, name): return self.ckpt_dir / f"{name}.pt"

    def save(self, name, epoch):
        torch.save({
            "epoch": epoch,
            "model_state_dict": self.model.state_dict(),
            "optim_state_dict": self.optim.state_dict(),
            "sched_state_dict": self.sched.state_dict(),
            "scaler_state_dict": self.scaler.state_dict(),
            "best_val_f1": self.best_val_f1,
            "history": self.history,
            "config": asdict(self.cfg),
        }, self._ckpt(name))

    def maybe_resume(self):
        last = self._ckpt("last")
        if not last.exists():
            return False
        print(f"[resume] loading {last}")
        p = torch.load(last, map_location=self.device, weights_only=False)
        self.model.load_state_dict(p["model_state_dict"])
        self.optim.load_state_dict(p["optim_state_dict"])
        self.sched.load_state_dict(p["sched_state_dict"])
        self.scaler.load_state_dict(p["scaler_state_dict"])
        self.start_epoch = p["epoch"] + 1
        self.best_val_f1 = p["best_val_f1"]; self.history = p["history"]
        print(f"[resume] from epoch {self.start_epoch}, best F1 so far {self.best_val_f1:.4f}")
        return True

    def _step(self, batch):
        batch = {k: v.to(self.device) if torch.is_tensor(v) else v for k, v in batch.items()}
        with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):
            out = self.model(**batch)
            losses = self.loss_fn(
                risk_logits=out["risk_logits"], cascade_logits=out["cascade_logits"],
                risk_labels=batch["risk_labels"], cascade_labels=batch["cascade_labels"],
                event_mask=batch["event_mask"],
            )
        return {"losses": losses, "out": out, "batch": batch}

    def train_epoch(self, epoch):
        self.model.train()
        if hasattr(self.model, "cfg") and getattr(self.model.cfg, "freeze_encoder", False):
            if hasattr(self.model, "encoder"):
                self.model.encoder.eval()
        running = {"total": 0, "risk": 0, "cascade": 0, "n": 0}
        pbar = tqdm(self.train_loader, desc=f"epoch {epoch+1}/{self.cfg.num_epochs}")
        self.optim.zero_grad(set_to_none=True)
        for step, batch in enumerate(pbar):
            r = self._step(batch)
            total = r["losses"]["total"] / self.cfg.grad_accum_steps
            self.scaler.scale(total).backward()
            if (step + 1) % self.cfg.grad_accum_steps == 0:
                self.scaler.unscale_(self.optim)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg.max_grad_norm)
                self.scaler.step(self.optim); self.scaler.update(); self.sched.step()
                self.optim.zero_grad(set_to_none=True)
            for k in ("total","risk","cascade"):
                running[k] += r["losses"][k].item()
            running["n"] += 1
            if step % 50 == 0:
                pbar.set_postfix(loss=running["total"] / max(running["n"], 1))
        return {k: running[k] / max(running["n"], 1) for k in ("total","risk","cascade")}

    @torch.no_grad()
    def evaluate(self, loader, desc="val"):
        self.model.eval()
        all_rp, all_rt, all_cp, all_ct, all_m = [], [], [], [], []
        rl = {"total": 0, "risk": 0, "cascade": 0, "n": 0}
        for batch in tqdm(loader, desc=desc):
            r = self._step(batch)
            for k in ("total","risk","cascade"):
                rl[k] += r["losses"][k].item()
            rl["n"] += 1
            rp = r["out"]["risk_logits"].argmax(-1).cpu().numpy()
            cp = torch.sigmoid(r["out"]["cascade_logits"]).cpu().numpy()
            m = r["batch"]["event_mask"].cpu().numpy().astype(bool)
            rt = r["batch"]["risk_labels"].cpu().numpy()
            ct = r["batch"]["cascade_labels"].cpu().numpy()
            all_rp.append(rp); all_rt.append(rt); all_cp.append(cp); all_ct.append(ct); all_m.append(m)
        mask = np.concatenate([m.ravel() for m in all_m])
        rp = np.concatenate([a.ravel() for a in all_rp])[mask]
        rt = np.concatenate([a.ravel() for a in all_rt])[mask]
        cp = np.concatenate([a.ravel() for a in all_cp])[mask]
        ct = np.concatenate([a.ravel() for a in all_ct])[mask]
        try:
            auc = float(roc_auc_score(ct, cp))
        except ValueError:
            auc = float("nan")
        return {
            "loss_total": rl["total"] / max(rl["n"], 1),
            "risk_f1_macro": float(f1_score(rt, rp, average="macro", zero_division=0)),
            "risk_f1_weighted": float(f1_score(rt, rp, average="weighted", zero_division=0)),
            "risk_accuracy": float(accuracy_score(rt, rp)),
            "cascade_auc": auc,
            "cascade_f1": float(f1_score(ct, (cp >= 0.5).astype(int), zero_division=0)),
            "_risk_pred": rp, "_risk_true": rt, "_cascade_prob": cp, "_cascade_true": ct,
        }

    def fit(self):
        self.maybe_resume()
        for epoch in range(self.start_epoch, self.cfg.num_epochs):
            t0 = time.time()
            tr = self.train_epoch(epoch)
            va = self.evaluate(self.val_loader, desc="val")
            va_log = {k: v for k, v in va.items() if not k.startswith("_")}
            entry = {"epoch": epoch, "train": tr, "val": va_log, "elapsed_sec": time.time() - t0}
            self.history.append(entry)
            print(f"[e{epoch+1}] train_loss={tr['total']:.4f} val_f1={va['risk_f1_macro']:.4f} "
                  f"val_cascade_auc={va['cascade_auc']:.4f} ({entry['elapsed_sec']/60:.1f} min)")
            self.save("last", epoch)
            if va["risk_f1_macro"] > self.best_val_f1:
                self.best_val_f1 = va["risk_f1_macro"]
                self.save("best", epoch); self.patience_counter = 0
                print(f"  ✓ new best F1={self.best_val_f1:.4f} saved")
            else:
                self.patience_counter += 1
                if self.patience_counter >= self.cfg.patience:
                    print(f"  early stop after {self.patience_counter} bad epochs")
                    break
            with open(self.ckpt_dir / "history.json", "w") as f:
                json.dump(self.history, f, indent=2)
        return {"best_val_f1": self.best_val_f1, "history": self.history}


print("pstgen_lib ready.")


Writing /content/pstgen_lib.py


In [ ]:
# Load the library
import sys
if '/content' not in sys.path:
    sys.path.insert(0, '/content')
import importlib
import pstgen_lib
importlib.reload(pstgen_lib)
from pstgen_lib import (
    ArabicTextPreprocessor, RegisterAdaptedHeuristics,
    PSTGenConfig, PSTGenGenerator,
    TimelineDataset, build_loaders,
    BiLSTMConfig, BiLSTMCascadeDetector,
    MonoBERTConfig, MonoBERTCascadeDetector,
    TCDTConfig, TCDTModel,
    MambaEventConfig, MambaEventModel,
    TrainConfig, CascadeLoss, Trainer,
)
print('✓ library loaded')

pstgen_lib ready.
pstgen_lib ready.
✓ library loaded


---
## 4. Load and normalize source corpora

In [ ]:
# Load real data from disk and normalize
import json
import pandas as pd
from pathlib import Path

RAW = Path('/content/pstgen/raw')

# Load AHD (real medinstruct-52k-arabic)
df_ahd = pd.read_csv(RAW / 'AHD.csv')
health_raw = df_ahd['Question'].dropna().astype(str).tolist()
print(f'[health] loaded {len(health_raw):,} rows from {RAW / "AHD.csv"}')
print(f'[health] sample: {health_raw[0][:120]}')

# Load SOQAL
with open(RAW / 'Arabic-SQuAD.json', 'r', encoding='utf-8') as f:
    soqal = json.load(f)
edu_raw = []
for article in soqal['data']:
    for para in article['paragraphs']:
        if para.get('context'):
            edu_raw.append(para['context'])
        for qa in para.get('qas', []):
            if qa.get('question'):
                edu_raw.append(qa['question'])

# Add ARCD for extra variety
if (RAW / 'ARCD.json').exists():
    with open(RAW / 'ARCD.json', 'r', encoding='utf-8') as f:
        arcd = json.load(f)
    for article in arcd['data']:
        for para in article['paragraphs']:
            if para.get('context'):
                edu_raw.append(para['context'])
            for qa in para.get('qas', []):
                if qa.get('question'):
                    edu_raw.append(qa['question'])

print(f'[edu] loaded {len(edu_raw):,} texts (SOQAL + ARCD)')
print(f'[edu] sample: {edu_raw[0][:120]}')

# Normalize
pp = ArabicTextPreprocessor(min_length=20, max_length=512)
print('\n[clean] normalizing...')
health_clean = pp.filter_and_normalize(health_raw)
edu_clean = pp.filter_and_normalize(edu_raw)
print(f'[clean] health: {len(health_raw):,} → {len(health_clean):,}')
print(f'[clean] edu:    {len(edu_raw):,} → {len(edu_clean):,}')


[health] loaded 52,002 rows from /content/pstgen/raw/AHD.csv
[health] sample: اشرح لماذا يمكن أن تؤدي الكتلة الموجودة في الرئة إلى ضيق التنفس. <noinput>
[edu] loaded 60,568 texts (SOQAL + ARCD)
[edu] sample: يعتمد ASCII أساس ا على الأبجدية الإنجليزية ، ويقوم بترميز 128 حرف ا محدد ا في أعداد صحيحة من سبعة أجزاء كما هو موضح في م

[clean] normalizing...
[clean] health: 52,002 → 50,510
[clean] edu:    60,568 → 53,219


---
## 5. Register-adapted risk labeling

In [ ]:
# Apply register-adapted labeling
from collections import defaultdict
from tqdm.auto import tqdm

heur = RegisterAdaptedHeuristics()

health_by_label = defaultdict(list)
edu_by_label = defaultdict(list)

print('[label] labeling health...')
for t in tqdm(health_clean):
    r = heur.label(t, 'Health')
    health_by_label[r].append(t)

print('[label] labeling edu...')
for t in tqdm(edu_clean):
    r = heur.label(t, 'Education')
    edu_by_label[r].append(t)

# Report — use len(v) since v is a list of texts
h_total = sum(len(v) for v in health_by_label.values())
e_total = sum(len(v) for v in edu_by_label.values())

print('\nHealth label pools:')
for k, v in sorted(health_by_label.items()):
    print(f'  {k:20s} {len(v):>8,}  ({100*len(v)/h_total:5.2f}%)')
print('\nEdu label pools:')
for k, v in sorted(edu_by_label.items()):
    print(f'  {k:20s} {len(v):>8,}  ({100*len(v)/e_total:5.2f}%)')

# Sample labeled texts
import random
random.seed(42)
print('\n--- Sample labeled texts ---')
for domain_label, pool in [('Health/NoRisk', health_by_label.get('NoRisk', [])),
                            ('Health/HealthConcern', health_by_label.get('HealthConcern', [])),
                            ('Edu/NoRisk', edu_by_label.get('NoRisk', [])),
                            ('Edu/EduDisengage', edu_by_label.get('EduDisengage', []))]:
    print(f'\n{domain_label} (showing 2):')
    if pool:
        for t in random.sample(pool, min(2, len(pool))):
            print(f'  {t[:150]}')
    else:
        print('  (empty pool!)')

[label] labeling health...


  0%|          | 0/50510 [00:00<?, ?it/s]

[label] labeling edu...


  0%|          | 0/53219 [00:00<?, ?it/s]


Health label pools:
  HealthConcern           5,956  (11.79%)
  NoRisk                 44,554  (88.21%)

Edu label pools:
  EduDisengage              186  ( 0.35%)
  NoRisk                 53,033  (99.65%)

--- Sample labeled texts ---

Health/NoRisk (showing 2):
  كيف تفسر هذه البيانات المتعلقه بانتشار جائحه كوفيد-19؟ "يظهر التقرير الاسبوعي لكوفيد-19 ارتفاعا بنسبه 20% في حالات الاصابه، وانخفاضا في الوفيات بنسبه 
  يرجي تلخيص مراجعه الادويه التاليه في فقره موجزه مناسبه للاشخاص العاديين لفهمها. الباراسيتامول هو دواء شائع لا يستلزم وصفه طبيه ويستخدم عالميا لتخفيف ا

Health/HealthConcern (showing 2):
  مع مجموعات متعدده من اعراض المريض، ابحث عن الامراض الاساسيه الشائعه لهذه المجموعات. المجموعه 1: حمي مستمره، تعرق ليلي، تضخم الغدد الليمفاويه. المجموعه
  قم بتصنيف حاله الصحه العقليه هذه وفقا لمعايير DSM-V. يعاني رجل يبلغ من العمر 27 عاما من تقلبات شديده لمده عام تقريبا، بدءا من فترات النشوه الشديده او 

Edu/NoRisk (showing 2):
  ما كان يسمي مارفيل اتفاقيه الكتاب الهزلي استضافت؟
  كم عدد ال

---
## 6. Generate synthetic timelines

This wipes any stale timelines and regenerates with real data.


In [ ]:
# Regenerate timelines. Wipes stale file first.
import json
from pathlib import Path
from collections import Counter

timelines_path = Path('/content/pstgen/synthetic/timelines.json')

# Always regenerate — if you trust an existing file, comment out the unlink
if timelines_path.exists():
    timelines_path.unlink()
    print(f'✓ deleted stale {timelines_path}')

cfg = PSTGenConfig(
    num_timelines=10_000,
    events_per_timeline=20,
    cascade_prob_in_window=0.25,  # higher than before — more cascade signal
    seed=42,
)
gen = PSTGenGenerator(dict(health_by_label), dict(edu_by_label), cfg)
timelines = gen.generate()

with open(timelines_path, 'w', encoding='utf-8') as f:
    json.dump(timelines, f, ensure_ascii=False)

size_mb = timelines_path.stat().st_size / 1e6
print(f'\n✓ saved {len(timelines):,} timelines to {timelines_path} ({size_mb:.1f} MB)')

# Sanity stats
risk_counts = Counter(e['risk_label'] for t in timelines for e in t['events'])
cascade_events = sum(1 for t in timelines for e in t['events'] if e['cascade'])
cascade_tls = sum(1 for t in timelines if any(e['cascade'] for e in t['events']))
n_events = sum(len(t['events']) for t in timelines)
print(f'  total events:               {n_events:,}')
print(f'  cascade events:             {cascade_events:,} ({100*cascade_events/n_events:.3f}%)')
print(f'  timelines with ≥1 cascade:  {cascade_tls:,} ({100*cascade_tls/len(timelines):.2f}%)')
print(f'  risk distribution: {dict(risk_counts)}')


✓ deleted stale /content/pstgen/synthetic/timelines.json


generating:   0%|          | 0/10000 [00:00<?, ?it/s]


✓ saved 10,000 timelines to /content/pstgen/synthetic/timelines.json (71.1 MB)
  total events:               200,000
  cascade events:             1,355 (0.677%)
  timelines with ≥1 cascade:  1,149 (11.49%)
  risk distribution: {'NoRisk': 165955, 'HealthConcern': 28905, 'EduDisengage': 5140}


---
## 7. Build train/val/test loaders (using AraModernBERT tokenizer)

In [ ]:
# Build loaders. Tokenizer is AraModernBERT (the 2026 upgrade).
# If this is slow on first run, it's downloading the tokenizer (~80 MB).
import torch

ds = TimelineDataset(
    timelines_path,
    tokenizer_name='NAMAA-Space/AraModernBert-Base-V1.0',
    max_events=20,
    max_tokens_per_event=128,
)
train_loader, val_loader, test_loader = build_loaders(ds, batch_size=16, seed=42)
print(f'✓ dataset ready: {len(ds):,} timelines')
print(f'  train: {len(train_loader.dataset):,}')
print(f'  val:   {len(val_loader.dataset):,}')
print(f'  test:  {len(test_loader.dataset):,}')

# Quick batch sanity check
batch = next(iter(train_loader))
print(f'\nbatch shapes:')
for k, v in batch.items():
    if torch.is_tensor(v):
        print(f'  {k:15s} {tuple(v.shape)}  {v.dtype}')
    else:
        print(f'  {k:15s} (list of {len(v)})')


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✓ dataset ready: 10,000 timelines
  train: 7,000
  val:   1,500
  test:  1,500

batch shapes:
  input_ids       (16, 20, 128)  torch.int64
  attention_mask  (16, 20, 128)  torch.int64
  domains         (16, 20)  torch.int64
  risk_labels     (16, 20)  torch.int64
  cascade_labels  (16, 20)  torch.float32
  event_mask      (16, 20)  torch.bool
  citizen_id      (list of 16)


# **Train  BiLSTM**

In [ ]:
# Check current state
from pathlib import Path
try:
    print('train_loader:', len(train_loader.dataset) if 'train_loader' in dir() else 'MISSING')
except NameError:
    print('train_loader: MISSING')
try:
    print('BiLSTMConfig defined:', BiLSTMConfig is not None)
except NameError:
    print('library: MISSING')

# Check timelines exist
tp = Path('/content/pstgen/synthetic/timelines.json')
print(f'timelines.json: {"exists" if tp.exists() else "MISSING"} ({tp.stat().st_size/1e6:.1f} MB)' if tp.exists() else 'timelines.json: MISSING')

train_loader: 7000
BiLSTMConfig defined: True
timelines.json: exists (71.1 MB)


In [ ]:
# Train BiLSTM (hero model) with AraModernBERT. Retrains from scratch since checkpoint was lost.
import shutil
from pathlib import Path

# Wipe any stale directory
ckpt_dir = Path('/content/pstgen/checkpoints/bilstm')
if ckpt_dir.exists():
    shutil.rmtree(ckpt_dir)
    print('✓ wiped old bilstm dir')

bilstm_cfg = BiLSTMConfig(
    encoder_name='NAMAA-Space/AraModernBert-Base-V1.0',
    hidden_dim=128, num_layers=2, dropout=0.3, freeze_encoder=True,
)
model = BiLSTMCascadeDetector(bilstm_cfg)
print('BiLSTM params:', model.count_parameters())

trainer_cfg = TrainConfig(
    model_name='bilstm', num_epochs=15, learning_rate=3e-5,
    cascade_loss_weight=1.5, patience=3, grad_accum_steps=1,
    mixed_precision=True,
    checkpoint_dir='/content/pstgen/checkpoints/bilstm',
)
trainer = Trainer(model, train_loader, val_loader, trainer_cfg)
summary = trainer.fit()
print(f'\n✓ BiLSTM best val F1: {summary["best_val_f1"]:.4f}')

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: NAMAA-Space/AraModernBert-Base-V1.0
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BiLSTM params: {'total': 150263044, 'trainable': 1316356, 'frozen': 148946688}


/content/pstgen_lib.py:654: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=cfg.mixed_precision and self.device.type == "cuda")


epoch 1/15:   0%|          | 0/438 [00:00<?, ?it/s]

...

[e12] train_loss=0.2705 val_f1=0.7868 val_cascade_auc=0.9134 (10.5 min)


epoch 13/15:   0%|          | 0/438 [00:00<?, ?it/s]

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):


val:   0%|          | 0/94 [00:00<?, ?it/s]

[e13] train_loss=0.2679 val_f1=0.7914 val_cascade_auc=0.9137 (10.5 min)
  early stop after 3 bad epochs

✓ BiLSTM best val F1: 0.7960


In [ ]:
import torch, json, io, requests, numpy as np, pandas as pd
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoTokenizer

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load best checkpoint
bpath = Path('/content/pstgen/checkpoints/bilstm/best.pt')
p = torch.load(bpath, map_location=device, weights_only=False)
model.load_state_dict(p['model_state_dict'])
model.to(device).eval()
print(f'✓ BiLSTM loaded: epoch {p["epoch"]+1}, val_f1={p["best_val_f1"]:.4f}')

# Synthetic test eval
print('\n=== BiLSTM TEST ===')
bilstm_test = trainer.evaluate(test_loader, desc='bilstm test')
bilstm_log = {k: v for k, v in bilstm_test.items() if not k.startswith('_')}
print(json.dumps(bilstm_log, indent=2))
with open('/content/pstgen/results/bilstm_test.json', 'w') as f:
    json.dump(bilstm_log, f, indent=2)


# Real-data eval helper
CARMA_TO_RISK = {"anxiety":1,"depression":1,"autism":1,"bipolar":1,"ocd":1,
                 "ptsd":1,"schizophrenia":1,"adhd":1,
                 "control":0,"normal":0,"none":0}

@torch.no_grad()
def eval_on_real(model, texts, batch_size=16, max_tokens=128, max_events=20):
    tok = AutoTokenizer.from_pretrained('NAMAA-Space/AraModernBert-Base-V1.0')
    enc = tok(list(texts), padding="max_length", truncation=True,
              max_length=max_tokens, return_tensors="pt")
    N = len(texts)
    input_ids = torch.zeros(N, max_events, max_tokens, dtype=torch.long)
    attention_mask = torch.zeros(N, max_events, max_tokens, dtype=torch.long)
    input_ids[:,0,:] = enc["input_ids"]
    attention_mask[:,0,:] = enc["attention_mask"]
    preds = np.zeros(N, dtype=int); probs = np.zeros(N, dtype=float)
    for i in tqdm(range(0, N, batch_size), desc="eval"):
        ids = input_ids[i:i+batch_size].to(device)
        mask = attention_mask[i:i+batch_size].to(device)
        out = model(input_ids=ids, attention_mask=mask)
        pp = torch.softmax(out["risk_logits"][:,0,:], dim=-1).cpu().numpy()
        preds[i:i+batch_size] = (pp.argmax(-1) == 1).astype(int)
        probs[i:i+batch_size] = pp[:,1]
    return preds, probs


# CARMA
print('\n' + '='*60); print('CARMA (BiLSTM)'); print('='*60)
try:
    from datasets import load_dataset
    ds_h = load_dataset("smankarious/carma")
    df = ds_h['train'].to_pandas().rename(columns={'selftext':'text'})
    if 'is_arabic' in df.columns:
        df = df[df['is_arabic']==True].copy()
    df["risk_label"] = df["condition"].astype(str).str.lower().str.strip().map(CARMA_TO_RISK)
    df = df.dropna(subset=["risk_label"]).copy()
    df["risk_label"] = df["risk_label"].astype(int)
    df = df[df['text'].astype(str).str.len() > 5].reset_index(drop=True)
    print(f"samples: {len(df)}, labels: {df['risk_label'].value_counts().to_dict()}")
    preds, probs = eval_on_real(model, df["text"].tolist())
    y = df["risk_label"].to_numpy()
    carma = {
        "model": "bilstm",
        "n_samples": int(len(df)),
        "f1_macro": float(f1_score(y, preds, average="macro", zero_division=0)),
        "f1_healthconcern": float(f1_score(y, preds, pos_label=1, zero_division=0)),
        "accuracy": float(accuracy_score(y, preds)),
        "confusion_matrix": confusion_matrix(y, preds).tolist(),
    }
    print('\n=== CARMA RESULTS (BiLSTM) ===')
    for k,v in carma.items(): print(f"  {k}: {v}")
    with open("/content/pstgen/results/carma_eval_bilstm.json","w") as f:
        json.dump(carma, f, indent=2)
except Exception as e:
    print(f'failed: {type(e).__name__}: {e}')


# MentalQA
print('\n' + '='*60); print('MentalQA (BiLSTM)'); print('='*60)
try:
    url = "https://raw.githubusercontent.com/hasanhuz/MentalQA/main/Train_Dev.tsv"
    df_mqa = pd.read_csv(io.StringIO(requests.get(url, timeout=30).text), sep="\t")
    texts = df_mqa['question'].fillna("").astype(str).tolist()
    texts = [t for t in texts if len(t) > 5]
    preds, probs = eval_on_real(model, texts)
    mqa = {
        "model": "bilstm",
        "n_samples": int(len(texts)),
        "note": "All MentalQA rows are mental-health Qs → ground truth uniformly HealthConcern",
        "recall_healthconcern": float((preds == 1).mean()),
        "mean_healthconcern_prob": float(probs.mean()),
    }
    print('\n=== MentalQA RESULTS (BiLSTM) ===')
    for k,v in mqa.items(): print(f"  {k}: {v}")
    with open("/content/pstgen/results/mentalqa_eval_bilstm.json","w") as f:
        json.dump(mqa, f, indent=2)
except Exception as e:
    print(f'failed: {type(e).__name__}: {e}')

✓ BiLSTM loaded: epoch 10, val_f1=0.7960

=== BiLSTM TEST ===


bilstm test:   0%|          | 0/94 [00:00<?, ?it/s]

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):


{
  "loss_total": 0.2527607305252806,
  "risk_f1_macro": 0.7971506705107658,
  "risk_f1_weighted": 0.8962736200190858,
  "risk_accuracy": 0.8892666666666666,
  "cascade_auc": 0.9108717084633442,
  "cascade_f1": 0.0
}

CARMA (BiLSTM)


README.md: 0.00B [00:00, ?B/s]

...

samples: 98, labels: {0: 72, 1: 26}


eval:   0%|          | 0/7 [00:00<?, ?it/s]


=== CARMA RESULTS (BiLSTM) ===
  model: bilstm
  n_samples: 98
  f1_macro: 0.49411521784018175
  f1_healthconcern: 0.13793103448275862
  accuracy: 0.7448979591836735
  confusion_matrix: [[71, 1], [24, 2]]

MentalQA (BiLSTM)


eval:   0%|          | 0/22 [00:00<?, ?it/s]


=== MentalQA RESULTS (BiLSTM) ===
  model: bilstm
  n_samples: 350
  note: All MentalQA rows are mental-health Qs → ground truth uniformly HealthConcern
  recall_healthconcern: 0.8
  mean_healthconcern_prob: 0.7546552616277976


---
## 8. Train AraModernBERT + BiLSTM (HERO MODEL)

Expected: val_f1 ≈ 0.85–0.92, cascade_auc > 0.95. ~1 hour on T4.


In [ ]:
# Wipe stale checkpoints from previous runs to avoid confusion
import shutil
from pathlib import Path
ckpt_dir = Path('/content/pstgen/checkpoints/bilstm')
if ckpt_dir.exists():
    # Only wipe if the resume would be from a different encoder
    try:
        last = ckpt_dir / 'last.pt'
        if last.exists():
            import torch
            p = torch.load(last, map_location='cpu', weights_only=False)
            # Check encoder name in config
            cfg_dict = p.get('config', {})
            # If we can't tell, just wipe to be safe
            shutil.rmtree(ckpt_dir)
            print(f'✓ wiped stale checkpoints (fresh start with AraModernBERT)')
    except Exception:
        shutil.rmtree(ckpt_dir)
        print(f'✓ wiped stale checkpoints')

✓ wiped stale checkpoints (fresh start with AraModernBERT)


In [ ]:
# Train the hero BiLSTM with AraModernBERT encoder
bilstm_cfg = BiLSTMConfig(
    encoder_name='NAMAA-Space/AraModernBert-Base-V1.0',
    hidden_dim=128, num_layers=2, dropout=0.3, freeze_encoder=True,
)
model = BiLSTMCascadeDetector(bilstm_cfg)
print('BiLSTM params:', model.count_parameters())

trainer_cfg = TrainConfig(
    model_name='bilstm', num_epochs=15, learning_rate=3e-5,
    cascade_loss_weight=1.5, patience=3, grad_accum_steps=1,
    mixed_precision=True,
    checkpoint_dir='/content/pstgen/checkpoints/bilstm',
)
trainer = Trainer(model, train_loader, val_loader, trainer_cfg)
summary = trainer.fit()
print(f'\n✓ BiLSTM best val F1: {summary["best_val_f1"]:.4f}')


model.safetensors:   0%|          | 0.00/598M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: NAMAA-Space/AraModernBert-Base-V1.0
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BiLSTM params: {'total': 150263044, 'trainable': 1316356, 'frozen': 148946688}


/content/pstgen_lib.py:654: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=cfg.mixed_precision and self.device.type == "cuda")


...

[e14] train_loss=0.2635 val_f1=0.8001 val_cascade_auc=0.9169 (10.2 min)


epoch 15/15:   0%|          | 0/438 [00:00<?, ?it/s]

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):


val:   0%|          | 0/94 [00:00<?, ?it/s]

[e15] train_loss=0.2625 val_f1=0.7999 val_cascade_auc=0.9170 (10.1 min)

✓ BiLSTM best val F1: 0.8002


In [ ]:
# Evaluate BiLSTM on test set
import torch, json
from pathlib import Path
bpath = Path('/content/pstgen/checkpoints/bilstm/best.pt')
if bpath.exists():
    p = torch.load(bpath, map_location=trainer.device, weights_only=False)
    model.load_state_dict(p['model_state_dict'])
    print(f'loaded best checkpoint from epoch {p["epoch"]+1}')

bilstm_test = trainer.evaluate(test_loader, desc='bilstm test')
bilstm_log = {k: v for k, v in bilstm_test.items() if not k.startswith('_')}
print('\n=== BiLSTM TEST ===')
print(json.dumps(bilstm_log, indent=2))
with open('/content/pstgen/results/bilstm_test.json', 'w') as f:
    json.dump(bilstm_log, f, indent=2)


loaded best checkpoint from epoch 13


bilstm test:   0%|          | 0/94 [00:00<?, ?it/s]

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):



=== BiLSTM TEST ===
{
  "loss_total": 0.23954799961536488,
  "risk_f1_macro": 0.8008986414540931,
  "risk_f1_weighted": 0.8969348307361023,
  "risk_accuracy": 0.8896,
  "cascade_auc": 0.9119073866787617,
  "cascade_f1": 0.008658008658008658
}


---
## 9. Train AraModernBERT + Transformer (TCDT baseline)

~1 hour on T4.


In [ ]:
# Train TCDT
tcdt_cfg = TCDTConfig(
    encoder_name='NAMAA-Space/AraModernBert-Base-V1.0',
    num_heads=4, ff_dim=1536, num_layers=1, freeze_encoder=True,
)
tcdt_model = TCDTModel(tcdt_cfg)
print('TCDT params:', tcdt_model.count_parameters())

tcdt_trainer_cfg = TrainConfig(
    model_name='tcdt', num_epochs=15, learning_rate=3e-5,
    cascade_loss_weight=1.5, patience=3,
    mixed_precision=True,
    checkpoint_dir='/content/pstgen/checkpoints/tcdt',
)
tcdt_trainer = Trainer(tcdt_model, train_loader, val_loader, tcdt_trainer_cfg)
tcdt_summary = tcdt_trainer.fit()

# Test
import torch, json
from pathlib import Path
bpath = Path('/content/pstgen/checkpoints/tcdt/best.pt')
if bpath.exists():
    p = torch.load(bpath, map_location=tcdt_trainer.device, weights_only=False)
    tcdt_model.load_state_dict(p['model_state_dict'])
tcdt_test = tcdt_trainer.evaluate(test_loader, desc='tcdt test')
tcdt_log = {k: v for k, v in tcdt_test.items() if not k.startswith('_')}
print('\n=== TCDT TEST ===')
print(json.dumps(tcdt_log, indent=2))
with open('/content/pstgen/results/tcdt_test.json', 'w') as f:
    json.dump(tcdt_log, f, indent=2)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: NAMAA-Space/AraModernBert-Base-V1.0
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TCDT params: {'total': 153692164, 'trainable': 4745476, 'frozen': 148946688}


/content/pstgen_lib.py:654: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=cfg.mixed_precision and self.device.type == "cuda")


epoch 1/15:   0%|          | 0/438 [00:00<?, ?it/s]

...

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):


val:   0%|          | 0/94 [00:00<?, ?it/s]

[e14] train_loss=0.2494 val_f1=0.7924 val_cascade_auc=0.9211 (10.3 min)
  ✓ new best F1=0.7924 saved


epoch 15/15:   0%|          | 0/438 [00:00<?, ?it/s]

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):


In [ ]:
from pathlib import Path
import torch

for model_name in ['bilstm', 'tcdt', 'mamba', 'monobert']:
    ckpt_dir = Path(f'/content/drive/MyDrive/pstgen/checkpoints/{model_name}')
    best = ckpt_dir / 'best.pt'
    last = ckpt_dir / 'last.pt'
    print(f'\n{model_name}:')
    for name, p in [('best', best), ('last', last)]:
        if p.exists():
            try:
                ckpt = torch.load(p, map_location='cpu', weights_only=False)
                print(f"  {name}.pt: epoch {ckpt['epoch']+1}, "
                      f"best_val_f1={ckpt.get('best_val_f1', 'N/A'):.4f}")
            except Exception as e:
                print(f"  {name}.pt: exists but can't load: {e}")
        else:
            print(f"  {name}.pt: MISSING")


bilstm:
  best.pt: MISSING
  last.pt: MISSING

tcdt:
  best.pt: epoch 14, best_val_f1=0.7924
  last.pt: epoch 14, best_val_f1=0.7902

mamba:
  best.pt: MISSING
  last.pt: MISSING

monobert:
  best.pt: MISSING
  last.pt: MISSING


In [ ]:
# Evaluate TCDT on test set. Epoch 15 was interrupted; best is epoch 14.
# No need to retrain — just load and evaluate.
import torch, json
from pathlib import Path

tcdt_cfg = TCDTConfig(
    encoder_name='NAMAA-Space/AraModernBert-Base-V1.0',
    num_heads=4, ff_dim=1536, num_layers=1, freeze_encoder=True,
)
tcdt_model = TCDTModel(tcdt_cfg)

bpath = Path('/content/pstgen/checkpoints/tcdt/best.pt')
p = torch.load(bpath, map_location='cuda', weights_only=False)
tcdt_model.load_state_dict(p['model_state_dict'])
print(f'loaded TCDT best checkpoint from epoch {p["epoch"]+1}')

tcdt_trainer_cfg = TrainConfig(
    model_name='tcdt', num_epochs=1, learning_rate=3e-5,
    cascade_loss_weight=1.5, mixed_precision=True,
    checkpoint_dir='/content/pstgen/checkpoints/tcdt',
)
tcdt_trainer = Trainer(tcdt_model, train_loader, val_loader, tcdt_trainer_cfg)

tcdt_test = tcdt_trainer.evaluate(test_loader, desc='tcdt test')
tcdt_log = {k: v for k, v in tcdt_test.items() if not k.startswith('_')}
print('\n=== TCDT TEST ===')
print(json.dumps(tcdt_log, indent=2))
with open('/content/pstgen/results/tcdt_test.json', 'w') as f:
    json.dump(tcdt_log, f, indent=2)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: NAMAA-Space/AraModernBert-Base-V1.0
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


loaded TCDT best checkpoint from epoch 14


/content/pstgen_lib.py:654: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=cfg.mixed_precision and self.device.type == "cuda")


tcdt test:   0%|          | 0/94 [00:00<?, ?it/s]

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):



=== TCDT TEST ===
{
  "loss_total": 0.2630579734736301,
  "risk_f1_macro": 0.7957052204440828,
  "risk_f1_weighted": 0.8963229741433569,
  "risk_accuracy": 0.889,
  "cascade_auc": 0.9118471987497555,
  "cascade_f1": 0.008403361344537815
}


---
## 10. Train Mamba-style Event SSM (2024 modern architecture baseline)

This is our 2026 architectural upgrade — a state-space model over event embeddings. Implements the core selective SSM mechanism from Gu & Dao (2023) in pure PyTorch (no CUDA kernels required), suitable for T4 free Colab. ~1 hour.


In [ ]:
# Train Mamba event SSM
mamba_cfg = MambaEventConfig(
    encoder_name='NAMAA-Space/AraModernBert-Base-V1.0',
    d_state=16, expand=2, num_blocks=2, freeze_encoder=True,
)
mamba_model = MambaEventModel(mamba_cfg)
print('Mamba-SSM params:', mamba_model.count_parameters())

mamba_trainer_cfg = TrainConfig(
    model_name='mamba', num_epochs=15, learning_rate=3e-5,
    cascade_loss_weight=1.5, patience=3,
    mixed_precision=True,
    checkpoint_dir='/content/pstgen/checkpoints/mamba',
)
mamba_trainer = Trainer(mamba_model, train_loader, val_loader, mamba_trainer_cfg)
mamba_summary = mamba_trainer.fit()

# Test
import torch, json
from pathlib import Path
bpath = Path('/content/pstgen/checkpoints/mamba/best.pt')
if bpath.exists():
    p = torch.load(bpath, map_location=mamba_trainer.device, weights_only=False)
    mamba_model.load_state_dict(p['model_state_dict'])
mamba_test = mamba_trainer.evaluate(test_loader, desc='mamba test')
mamba_log = {k: v for k, v in mamba_test.items() if not k.startswith('_')}
print('\n=== MAMBA-SSM TEST ===')
print(json.dumps(mamba_log, indent=2))
with open('/content/pstgen/results/mamba_test.json', 'w') as f:
    json.dump(mamba_log, f, indent=2)


Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: NAMAA-Space/AraModernBert-Base-V1.0
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Mamba-SSM params: {'total': 160899844, 'trainable': 11953156, 'frozen': 148946688}


/content/pstgen_lib.py:654: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=cfg.mixed_precision and self.device.type == "cuda")


epoch 1/15:   0%|          | 0/438 [00:00<?, ?it/s]

...

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):


val:   0%|          | 0/94 [00:00<?, ?it/s]

[e6] train_loss=0.0509 val_f1=0.9759 val_cascade_auc=0.9387 (10.9 min)
  ✓ new best F1=0.9759 saved


epoch 7/15:   0%|          | 0/438 [00:00<?, ?it/s]

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):


In [ ]:
# Evaluate Mamba on the synthetic test set (it only had val F1 before)
import torch, json
from pathlib import Path

mamba_cfg = MambaEventConfig(
    encoder_name='NAMAA-Space/AraModernBert-Base-V1.0',
    d_state=16, expand=2, num_blocks=2, freeze_encoder=True,
)
mamba_model = MambaEventModel(mamba_cfg)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
bpath = Path('/content/pstgen/checkpoints/mamba/best.pt')
p = torch.load(bpath, map_location=device, weights_only=False)
mamba_model.load_state_dict(p['model_state_dict'])
mamba_model.to(device).eval()
print(f'✓ Mamba loaded: epoch {p["epoch"]+1}, val_f1={p["best_val_f1"]:.4f}')

# Build a temporary trainer just to use its evaluate() method
mamba_trainer_cfg = TrainConfig(
    model_name='mamba', num_epochs=1, learning_rate=3e-5,
    cascade_loss_weight=1.5, mixed_precision=True,
    checkpoint_dir='/content/pstgen/checkpoints/mamba',
)
mamba_trainer = Trainer(mamba_model, train_loader, val_loader, mamba_trainer_cfg)

# Test eval
mamba_test = mamba_trainer.evaluate(test_loader, desc='mamba test')
mamba_log = {k: v for k, v in mamba_test.items() if not k.startswith('_')}
print('\n=== MAMBA TEST ===')
print(json.dumps(mamba_log, indent=2))
with open('/content/pstgen/results/mamba_test.json', 'w') as f:
    json.dump(mamba_log, f, indent=2)

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: NAMAA-Space/AraModernBert-Base-V1.0
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Mamba loaded: epoch 5, val_f1=0.9731


/content/pstgen_lib.py:654: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=cfg.mixed_precision and self.device.type == "cuda")


mamba test:   0%|          | 0/94 [00:00<?, ?it/s]

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):



=== MAMBA TEST ===
{
  "loss_total": 0.09148047458221938,
  "risk_f1_macro": 0.9718200791271734,
  "risk_f1_weighted": 0.9812209791720037,
  "risk_accuracy": 0.9809666666666667,
  "cascade_auc": 0.938891715822619,
  "cascade_f1": 0.016877637130801686
}


---
## 11. Train AraModernBERT MonoBERT (non-temporal ablation)

Tests whether temporal modeling actually helps (spoiler: it should). ~45 min.


In [ ]:
# Train MonoBERT (non-temporal)
# Fine-tune the encoder here — MonoBERT with frozen encoder would be trivially weak
mono_cfg = MonoBERTConfig(
    encoder_name='NAMAA-Space/AraModernBert-Base-V1.0',
)
mono_model = MonoBERTCascadeDetector(mono_cfg)
print('MonoBERT params:', mono_model.count_parameters())

mono_trainer_cfg = TrainConfig(
    model_name='monobert', num_epochs=10, learning_rate=2e-5,
    cascade_loss_weight=1.5, patience=3, grad_accum_steps=2,
    mixed_precision=True,
    checkpoint_dir='/content/pstgen/checkpoints/monobert',
)
mono_trainer = Trainer(mono_model, train_loader, val_loader, mono_trainer_cfg)
mono_summary = mono_trainer.fit()

# Test
import torch, json
from pathlib import Path
bpath = Path('/content/pstgen/checkpoints/monobert/best.pt')
if bpath.exists():
    p = torch.load(bpath, map_location=mono_trainer.device, weights_only=False)
    mono_model.load_state_dict(p['model_state_dict'])
mono_test = mono_trainer.evaluate(test_loader, desc='monobert test')
mono_log = {k: v for k, v in mono_test.items() if not k.startswith('_')}
print('\n=== MONOBERT TEST ===')
print(json.dumps(mono_log, indent=2))
with open('/content/pstgen/results/monobert_test.json', 'w') as f:
    json.dump(mono_log, f, indent=2)

---
## 12. CARMA real-data validation

Evaluates the trained BiLSTM on CARMA (Arabic Reddit mental health, Mankarious et al. 2025). If CARMA's HuggingFace path has changed, this cell will fail gracefully and you can skip to MentalQA.


In [ ]:
# Load Mamba (best performer) for real-data evaluation
import torch
from pathlib import Path

mamba_cfg = MambaEventConfig(
    encoder_name='NAMAA-Space/AraModernBert-Base-V1.0',
    d_state=16, expand=2, num_blocks=2, freeze_encoder=True,
)
model = MambaEventModel(mamba_cfg)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
bpath = Path('/content/pstgen/checkpoints/mamba/best.pt')
p = torch.load(bpath, map_location=device, weights_only=False)
model.load_state_dict(p['model_state_dict'])
model.to(device).eval()
print(f'✓ Mamba loaded: epoch {p["epoch"]+1}, val_f1={p["best_val_f1"]:.4f}')

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: NAMAA-Space/AraModernBert-Base-V1.0
Key               | Status     |  | 
------------------+------------+--+-
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Mamba loaded: epoch 5, val_f1=0.9731


---
## 13. MentalQA real-data validation

In [ ]:
import numpy as np, pandas as pd, torch, requests, io, json
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoTokenizer

CARMA_TO_RISK = {
    "anxiety":1,"depression":1,"autism":1,"bipolar":1,"ocd":1,
    "ptsd":1,"schizophrenia":1,"adhd":1,
    "control":0,"normal":0,"none":0,
}

@torch.no_grad()
def eval_on_real(model, texts, tokenizer_name='NAMAA-Space/AraModernBert-Base-V1.0',
                 batch_size=16, max_tokens=128, max_events=20):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    tok = AutoTokenizer.from_pretrained(tokenizer_name)
    enc = tok(list(texts), padding="max_length", truncation=True,
              max_length=max_tokens, return_tensors="pt")
    N = len(texts)
    input_ids = torch.zeros(N, max_events, max_tokens, dtype=torch.long)
    attention_mask = torch.zeros(N, max_events, max_tokens, dtype=torch.long)
    input_ids[:,0,:] = enc["input_ids"]
    attention_mask[:,0,:] = enc["attention_mask"]
    preds = np.zeros(N, dtype=int); probs = np.zeros(N, dtype=float)
    for i in tqdm(range(0, N, batch_size), desc="eval"):
        ids = input_ids[i:i+batch_size].to(device)
        mask = attention_mask[i:i+batch_size].to(device)
        out = model(input_ids=ids, attention_mask=mask)
        p = torch.softmax(out["risk_logits"][:,0,:], dim=-1).cpu().numpy()
        preds[i:i+batch_size] = (p.argmax(-1) == 1).astype(int)
        probs[i:i+batch_size] = p[:,1]
    return preds, probs


# CARMA
print('='*60); print('CARMA (Mamba)'); print('='*60)
try:
    from datasets import load_dataset
    ds = load_dataset("smankarious/carma")
    df = ds['train'].to_pandas().rename(columns={'selftext':'text'})
    if 'is_arabic' in df.columns:
        df = df[df['is_arabic']==True].copy()
    df["condition_norm"] = df["condition"].astype(str).str.lower().str.strip()
    df["risk_label"] = df["condition_norm"].map(CARMA_TO_RISK)
    df = df.dropna(subset=["risk_label"]).copy()
    df["risk_label"] = df["risk_label"].astype(int)
    df = df[df['text'].astype(str).str.len() > 5].reset_index(drop=True)
    print(f"samples: {len(df)}, labels: {df['risk_label'].value_counts().to_dict()}")
    preds, probs = eval_on_real(model, df["text"].tolist())
    y = df["risk_label"].to_numpy()
    carma = {
        "model": "mamba",
        "n_samples": int(len(df)),
        "f1_macro": float(f1_score(y, preds, average="macro", zero_division=0)),
        "f1_healthconcern": float(f1_score(y, preds, pos_label=1, zero_division=0)),
        "accuracy": float(accuracy_score(y, preds)),
        "confusion_matrix": confusion_matrix(y, preds).tolist(),
    }
    print("\n=== CARMA RESULTS ===")
    for k,v in carma.items():
        print(f"  {k}: {v}")
    with open("/content/pstgen/results/carma_eval.json","w") as f:
        json.dump(carma, f, indent=2)
except Exception as e:
    print(f"failed: {type(e).__name__}: {e}")


# MentalQA
print('\n'+'='*60); print('MentalQA (Mamba)'); print('='*60)
try:
    url = "https://raw.githubusercontent.com/hasanhuz/MentalQA/main/Train_Dev.tsv"
    df_mqa = pd.read_csv(io.StringIO(requests.get(url, timeout=30).text), sep="\t")
    print(f"rows: {len(df_mqa)}, columns: {df_mqa.columns.tolist()}")
    texts = df_mqa['question'].fillna("").astype(str).tolist()
    texts = [t for t in texts if len(t) > 5]
    preds, probs = eval_on_real(model, texts)
    mqa = {
        "model": "mamba",
        "n_samples": int(len(texts)),
        "note": "All MentalQA rows are mental-health Qs → ground truth uniformly HealthConcern",
        "recall_healthconcern": float((preds == 1).mean()),
        "mean_healthconcern_prob": float(probs.mean()),
    }
    print("\n=== MentalQA RESULTS ===")
    for k,v in mqa.items():
        print(f"  {k}: {v}")
    with open("/content/pstgen/results/mentalqa_eval.json","w") as f:
        json.dump(mqa, f, indent=2)
except Exception as e:
    print(f"failed: {type(e).__name__}: {e}")

CARMA (Mamba)
samples: 98, labels: {0: 72, 1: 26}


eval:   0%|          | 0/7 [00:00<?, ?it/s]


=== CARMA RESULTS ===
  model: mamba
  n_samples: 98
  f1_macro: 0.4235294117647059
  f1_healthconcern: 0.0
  accuracy: 0.7346938775510204
  confusion_matrix: [[72, 0], [26, 0]]

MentalQA (Mamba)
rows: 350, columns: ['question', 'answer', 'final_QT', 'final_AS']


eval:   0%|          | 0/22 [00:00<?, ?it/s]


=== MentalQA RESULTS ===
  model: mamba
  n_samples: 350
  note: All MentalQA rows are mental-health Qs → ground truth uniformly HealthConcern
  recall_healthconcern: 0.4657142857142857
  mean_healthconcern_prob: 0.4665236652526246


---
## 14. Compile final results table

In [ ]:
# Build comparison table from all *_test.json files
import json
from pathlib import Path
import pandas as pd

results_dir = Path('/content/pstgen/results')
rows = []
for name in ('bilstm_test.json', 'tcdt_test.json', 'mamba_test.json', 'monobert_test.json'):
    p = results_dir / name
    if not p.exists(): continue
    with open(p) as f:
        r = json.load(f)
    rows.append({
        'Model': name.replace('_test.json','').replace('_',' ').title(),
        'Risk F1 (macro)': r.get('risk_f1_macro'),
        'Risk F1 (weighted)': r.get('risk_f1_weighted'),
        'Accuracy': r.get('risk_accuracy'),
        'Cascade AUC': r.get('cascade_auc'),
        'Cascade F1': r.get('cascade_f1'),
    })

if rows:
    df = pd.DataFrame(rows).sort_values('Risk F1 (macro)', ascending=False)
    print('\n=== SYNTHETIC TEST SET RESULTS ===')
    print(df.to_string(index=False))
    df.to_csv(results_dir / 'summary_synthetic.csv', index=False)
else:
    print('no results found — finish training models first')

# Real-data results
print('\n=== REAL-DATA VALIDATION ===')
for name in ('carma_eval.json', 'mentalqa_eval.json'):
    p = results_dir / name
    if not p.exists():
        print(f'{name}: not yet run')
        continue
    with open(p) as f:
        r = json.load(f)
    print(f'\n{name}:')
    for k, v in r.items():
        if k in ('classification_report', 'confusion_matrix'):
            continue
        print(f'  {k}: {v}')



=== SYNTHETIC TEST SET RESULTS ===
 Model  Risk F1 (macro)  Risk F1 (weighted)  Accuracy  Cascade AUC  Cascade F1
Bilstm         0.800899            0.896935    0.8896     0.911907    0.008658
  Tcdt         0.795705            0.896323    0.8890     0.911847    0.008403

=== REAL-DATA VALIDATION ===

carma_eval.json:
  model: mamba
  n_samples: 98
  f1_macro: 0.4235294117647059
  f1_healthconcern: 0.0
  accuracy: 0.7346938775510204

mentalqa_eval.json:
  model: mamba
  n_samples: 350
  note: All MentalQA rows are mental-health Qs → ground truth uniformly HealthConcern
  recall_healthconcern: 0.4657142857142857
  mean_healthconcern_prob: 0.4665236652526246


In [ ]:
# Tune cascade threshold on validation set, apply to test set, for all 3 models.
# This fixes the cascade_f1 ≈ 0 issue caused by default threshold 0.5 on rare positives.

import torch, json, numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_curve

device = 'cuda' if torch.cuda.is_available() else 'cpu'

def tune_threshold(model, trainer, model_name, results_path):
    """Re-evaluate model on val + test, find best threshold on val, apply to test."""
    print(f'\n{"="*60}\n{model_name.upper()} threshold tuning\n{"="*60}')

    val_out = trainer.evaluate(val_loader, desc=f'{model_name} val')
    test_out = trainer.evaluate(test_loader, desc=f'{model_name} test')

    val_probs = val_out['_cascade_prob']
    val_trues = val_out['_cascade_true']
    test_probs = test_out['_cascade_prob']
    test_trues = test_out['_cascade_true']

    # Sweep thresholds on val
    precisions, recalls, thresholds = precision_recall_curve(val_trues, val_probs)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-10)
    best_idx = f1s[:-1].argmax()
    best_thr = float(thresholds[best_idx])

    print(f'  best threshold (val): {best_thr:.4f}')
    print(f'  val F1 @ tuned thr:   {f1s[best_idx]:.4f}')

    # Apply to test
    test_preds = (test_probs >= best_thr).astype(int)
    test_f1 = float(f1_score(test_trues, test_preds, zero_division=0))
    test_p = float(precision_score(test_trues, test_preds, zero_division=0))
    test_r = float(recall_score(test_trues, test_preds, zero_division=0))

    print(f'  test F1 @ tuned thr:  {test_f1:.4f}')
    print(f'  test precision:       {test_p:.4f}')
    print(f'  test recall:          {test_r:.4f}')
    print(f'  test positives:       {int(test_trues.sum())} of {len(test_trues)}')
    print(f'  predicted positives:  {int(test_preds.sum())}')

    # Update results file
    log = {k: v for k, v in test_out.items() if not k.startswith('_')}
    log['cascade_threshold_tuned'] = best_thr
    log['cascade_f1_tuned'] = test_f1
    log['cascade_precision_tuned'] = test_p
    log['cascade_recall_tuned'] = test_r

    with open(results_path, 'w') as f:
        json.dump(log, f, indent=2)
    print(f'  → saved to {results_path}')
    return log


# --- BiLSTM ---
bilstm_cfg = BiLSTMConfig(
    encoder_name='NAMAA-Space/AraModernBert-Base-V1.0',
    hidden_dim=128, num_layers=2, dropout=0.3, freeze_encoder=True,
)
bilstm_model = BiLSTMCascadeDetector(bilstm_cfg)
p = torch.load('/content/pstgen/checkpoints/bilstm/best.pt', map_location=device, weights_only=False)
bilstm_model.load_state_dict(p['model_state_dict'])
bilstm_model.to(device).eval()
bilstm_trainer = Trainer(bilstm_model, train_loader, val_loader,
                         TrainConfig(model_name='bilstm', num_epochs=1,
                                     checkpoint_dir='/content/pstgen/checkpoints/bilstm'))
bilstm_log = tune_threshold(bilstm_model, bilstm_trainer, 'bilstm',
                            '/content/pstgen/results/bilstm_test.json')


# --- TCDT ---
tcdt_cfg = TCDTConfig(
    encoder_name='NAMAA-Space/AraModernBert-Base-V1.0',
    num_heads=4, ff_dim=1536, num_layers=1, freeze_encoder=True,
)
tcdt_model = TCDTModel(tcdt_cfg)
p = torch.load('/content/pstgen/checkpoints/tcdt/best.pt', map_location=device, weights_only=False)
tcdt_model.load_state_dict(p['model_state_dict'])
tcdt_model.to(device).eval()
tcdt_trainer = Trainer(tcdt_model, train_loader, val_loader,
                       TrainConfig(model_name='tcdt', num_epochs=1,
                                   checkpoint_dir='/content/pstgen/checkpoints/tcdt'))
tcdt_log = tune_threshold(tcdt_model, tcdt_trainer, 'tcdt',
                          '/content/pstgen/results/tcdt_test.json')


# --- Mamba ---
mamba_log = tune_threshold(mamba_model, mamba_trainer, 'mamba',
                           '/content/pstgen/results/mamba_test.json')


# --- Final summary ---
print('\n\n' + '='*60)
print('FINAL SYNTHETIC TEST RESULTS (all models, tuned threshold)')
print('='*60)
print(f'{"Model":12s} {"Risk F1":>10s} {"Cascade AUC":>13s} {"Cascade F1":>12s} {"Threshold":>11s}')
for name, log in [('BiLSTM', bilstm_log), ('TCDT', tcdt_log), ('Mamba', mamba_log)]:
    print(f'{name:12s} {log["risk_f1_macro"]:>10.4f} {log["cascade_auc"]:>13.4f} '
          f'{log["cascade_f1_tuned"]:>12.4f} {log["cascade_threshold_tuned"]:>11.4f}')

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: NAMAA-Space/AraModernBert-Base-V1.0
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



BILSTM threshold tuning


/content/pstgen_lib.py:654: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=cfg.mixed_precision and self.device.type == "cuda")


bilstm val:   0%|          | 0/94 [00:00<?, ?it/s]

...

mamba val:   0%|          | 0/94 [00:00<?, ?it/s]

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):


mamba test:   0%|          | 0/94 [00:00<?, ?it/s]

/content/pstgen_lib.py:689: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=self.cfg.mixed_precision and self.device.type == "cuda"):


  best threshold (val): 0.2124
  val F1 @ tuned thr:   0.1570
  test F1 @ tuned thr:  0.1022
  test precision:       0.1301
  test recall:          0.0841
  test positives:       226 of 30000
  predicted positives:  146
  → saved to /content/pstgen/results/mamba_test.json


FINAL SYNTHETIC TEST RESULTS (all models, tuned threshold)
Model           Risk F1   Cascade AUC   Cascade F1   Threshold
BiLSTM           0.7972        0.9109       0.1614      0.0349
TCDT             0.7957        0.9118       0.1638      0.0477
Mamba            0.9718        0.9389       0.1022      0.2124


---
## 15. Backup to local machine

Run this at the end of every session.


In [ ]:
import os
from datetime import datetime
from google.colab import files

ts = datetime.now().strftime('%Y%m%d_%H%M')
zip_path = f'/content/pstgen_backup_{ts}.zip'

!cd /content/pstgen && zip -r {zip_path} \
    synthetic/ checkpoints/ results/ -x "**/__pycache__/*" 2>&1 | tail -5

size_mb = os.path.getsize(zip_path) / 1e6
print(f'\n✓ backup ready: {zip_path} ({size_mb:.1f} MB)')
files.download(zip_path)


---
## Done

**What you have after running this notebook:**

- `/content/pstgen/results/summary_synthetic.csv` — comparison table for 4 models
- `/content/pstgen/results/carma_eval.json` — CARMA real-data numbers
- `/content/pstgen/results/mentalqa_eval.json` — MentalQA real-data numbers
- `/content/pstgen/checkpoints/{bilstm,tcdt,mamba,monobert}/best.pt` — model weights

**Next step:** send me `summary_synthetic.csv` and `carma_eval.json`. I'll write the paper sections around your actual numbers.

**Note on Jais zero-shot:** Not included in this notebook because it needs careful memory management with 4-bit quantization. After all 4 trained models finish, I'll give you a separate short notebook for Jais-1.3B inference as the 5th baseline. This keeps the main run clean and avoids memory issues.
